## Figure 1


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

main_DISPLAY = {'Malmo': 'Malmö', 'Brasilia': 'Brasília'}

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_panel_heading(fig, x, y, letter, title):
    heading = fig.text(x, y, f"({letter.strip('()')}) {title}", fontweight='bold', fontsize=STYLE['title_size'], va='center', gid='panel-title')
    fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [heading]

def main_bubble_legend_check(study, name, legend, values, area_scale):
    """Validate legend areas using the same points-squared scale as the data."""
    observed = [float(handle.get_sizes()[0]) for handle in legend.legend_handles[:len(values)]]
    study.check(name + '_legend_areas', np.allclose(observed, np.asarray(values) * area_scale), {'values': values, 'areas_pt2': observed, 'area_scale': area_scale})
    study.check(name + '_legend_marker_scale', legend.markerscale == 1.0, legend.markerscale)

def main_equal_earth(longitude, latitude):
    lon, lat = np.broadcast_arrays(np.asarray(longitude), np.asarray(latitude))
    th = np.arcsin(np.sqrt(3) / 2 * np.sin(np.deg2rad(lat)))
    a1, a2, a3, a4 = (1.340264, -0.081106, 0.000893, 0.003796)
    x = np.deg2rad(lon) * np.cos(th) / (np.sqrt(3) / 2 * (a1 + 3 * a2 * th ** 2 + 7 * a3 * th ** 6 + 9 * a4 * th ** 8))
    y = th * (a1 + a2 * th ** 2 + a3 * th ** 6 + a4 * th ** 8)
    return (x, y)

def main_project(ring):
    a = np.asarray(ring)
    return np.column_stack(main_equal_earth(a[:, 0], a[:, 1]))

def main_map_base(ax, land, countries):
    rings = []
    for f in land['features']:
        g = f['geometry']
        gs = [g['coordinates']] if g['type'] == 'Polygon' else g['coordinates']
        rings.extend((main_project(v[0]) for v in gs))
    land_layer = PolyCollection(rings, facecolors='#F3F1ED', edgecolors='#929A9E', lw=0.4, zorder=1)
    land_layer.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.65, -0.65), alpha=0.16), pe.Normal()])
    ax.add_collection(land_layer)
    country_layer = PolyCollection([main_project(r) for f in countries['features'] for r in f['rings']], facecolors='#D9CFE5', edgecolors='#A79BB4', lw=0.4, zorder=2)
    country_layer.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.12), pe.Normal()])
    ax.add_collection(country_layer)
    for lon in range(-180, 181, 60):
        ax.plot(*main_equal_earth(np.full(146, lon), np.linspace(-60, 85, 146)), c='#D8DBDF', lw=0.4, ls=':', zorder=0)
    for lat in [-60, -30, 0, 30, 60]:
        ax.plot(*main_equal_earth(np.linspace(-180, 180, 361), np.full(361, lat)), c='#D8DBDF', lw=0.4, ls=':', zorder=0)
    boundary = np.vstack([main_project(np.column_stack([np.linspace(-180, 180, 361), np.full(361, -60)])), main_project(np.column_stack([np.full(146, 180), np.linspace(-60, 85, 146)])), main_project(np.column_stack([np.linspace(180, -180, 361), np.full(361, 85)])), main_project(np.column_stack([np.full(146, -180), np.linspace(85, -60, 146)]))])
    ax.add_patch(Polygon(boundary, closed=True, fc='#F5F8FA', ec='none', zorder=-3))
    ax.plot(*np.vstack([boundary, boundary[:1]]).T, c='#555B60', lw=0.65, zorder=3)
    ax.set(xlim=(boundary[:, 0].min(), boundary[:, 0].max()), ylim=(boundary[:, 1].min(), boundary[:, 1].max()))
    ax.set_aspect('equal')
    ax.set_axis_off()

def main_figure_one(s: main_Study):
    points = s.csv('CITY_MAP_POINTS.csv')
    countries = s.csv('COUNTRY_DEMAND_SUMMARY.csv')
    variance = s.csv('ASHRAE_NESTED_VARIANCE.csv')
    sens = s.csv('LEAVE_ONE_SOURCE_VARIANCE.csv')
    rooms = s.csv('SHARED_ROOM_SUMMARY.csv')
    climates = pd.DataFrame(s.json('GLOBAL_CLIMATES.json')['climates'])
    s.check('fig1_observation_count', points.record_count.sum() == countries.record_count.sum() == 33845, 33845)
    s.check('fig1_locations', len(points) == 49 and len(countries) == 15 and (len(climates) == 9), [len(points), len(countries), len(climates)])
    f = main_page(1)
    f.set_size_inches(7.6, 8.0)
    a = f.add_axes([0.065, 0.58, 0.82, 0.37])
    main_panel_heading(f, 0.151639453, 0.962, 'a', 'Global observations and simulation climates')
    main_map_base(a, s.json('ne_110m_land.geojson'), s.json('ne_110m_observed_countries.json'))
    cmap = LinearSegmentedColormap.from_list('demand', ['#365889', '#56A9BE', '#F4DE91', '#E77C5E', '#A92C46'])
    norm = Normalize(np.floor(points.demand_center_c.min()), np.ceil(points.demand_center_c.max()))
    area_scale = 360 / points.record_count.max()
    cloud = a.scatter(*main_equal_earth(points.longitude, points.latitude), s=area_scale * points.record_count, c=points.demand_center_c, cmap=cmap, norm=norm, edgecolor='white', lw=0.8, zorder=4)
    a.scatter(*main_equal_earth(climates.longitude, climates.latitude), marker='*', s=118, c=COLORS['gold'], edgecolors='#5E4A13', lw=0.7, zorder=6)
    labels = {'Miami': (-71, 19), 'Montreal': (-66, 50), 'Malmo': (27, 65), 'Tunis': (14, 29), 'Delhi': (86, 22), 'Beijing': (133, 44), 'Singapore': (127, -10), 'Sydney': (145, -46), 'Brasilia': (-39, -26)}
    for row in climates.itertuples():
        a.annotate(main_DISPLAY.get(row.climate, row.climate), main_equal_earth(row.longitude, row.latitude), xytext=main_equal_earth(*labels[row.climate]), ha='center', va='center', fontsize=7.6, arrowprops={'arrowstyle': '-', 'lw': 0.55, 'color': '#6B7075'}, zorder=7, bbox={'facecolor': 'white', 'alpha': 0.62, 'edgecolor': 'none', 'pad': 0.3})
    for latitude in [-60, -30, 0, 30, 60]:
        x, y = main_equal_earth(-180, latitude)
        label = '0°' if latitude == 0 else f'{abs(latitude)}°' + ('N' if latitude > 0 else 'S')
        a.text(x - 0.065, y, label, ha='right', va='center', clip_on=False)
    for longitude in [-180, -120, -60, 0, 60, 120, 180]:
        x, y = main_equal_earth(longitude, -60)
        label = '0°' if longitude == 0 else f'{abs(longitude)}°' + ('E' if longitude > 0 else 'W')
        a.text(x, y - 0.075, label, ha='center', va='top', clip_on=False)
    f.canvas.draw()
    box = a.get_position()
    cax = f.add_axes([0.906, box.y0, 0.018, box.height])
    cb = f.colorbar(cloud, cax=cax)
    cb.outline.set_linewidth(0.5)
    cax.set_title('Mean demand\ncenter (°C)', fontsize=8.7, fontweight='normal', pad=8)
    cb.ax.yaxis.set_major_locator(MultipleLocator(2))
    handles = [a.scatter([], [], s=area_scale * n, c='#9B9D9F', ec='white', lw=0.8) for n in [100, 500, 2000, 5000]]
    handles += [Patch(fc='#E4DFEC', ec='#AAA4B4', lw=0.5), Line2D([], [], ls='', marker='*', ms=8, c=COLORS['gold'], mec='#5E4A13', mew=0.6)]
    size_legend = f.legend(handles, ['100', '500', '2,000', '5,000 records', 'Countries with observations', '9 weather contexts'], loc='center', bbox_to_anchor=(0.51, 0.54), ncol=6, columnspacing=1.0, handletextpad=0.9, fontsize=7.2, markerscale=1)
    main_bubble_legend_check(s, 'fig1_records', size_legend, [100, 500, 2000, 5000], area_scale)
    b = main_axes(f, [0.151639453, 0.075, 0.3, 0.39], 'b', 'Demand within countries', 'x', letter_x=-0.49)
    cc = main_axes(f, [0.631639453, 0.295, 0.29, 0.17], 'c', 'Variation by scale', 'x', letter_x=-0.24)
    d = main_axes(f, [0.631639453, 0.075, 0.29, 0.105], 'd', '', 'x', letter_x=-0.24)
    main_panel_heading(f, 0.631639453, 0.225, 'd', 'Shared-room requests')
    countries = countries.sort_values('demand_center_median_c', ascending=False).reset_index(drop=True)
    names = countries.country.str.replace('_', ' ').str.title().replace({'Uk': 'United Kingdom', 'Usa': 'United States'})
    y = np.arange(15)
    for i, row in countries.iterrows():
        paint = cmap(norm(row.demand_center_median_c))
        b.plot([row.demand_center_p10_c, row.demand_center_p90_c], [i, i], c=paint, lw=1.8)
        b.plot([row.demand_center_p10_c, row.demand_center_p90_c], [i, i], ls='', marker='|', c=paint, ms=5, mew=1)
        b.scatter(row.demand_center_median_c, i, s=48, c=[paint], ec='white', lw=0.55, zorder=3)
        b.text(1.05, i, f'{int(row.record_count):,}', transform=b.get_yaxis_transform(), va='center', fontsize=7.8, clip_on=False)
    b.set(yticks=y, yticklabels=names, ylim=(14.7, -0.8), xlabel='Demand center (°C)', xlim=(18, 32.7))
    b.set_xticks([20, 24, 28, 32])
    b.tick_params(axis='y', length=0, pad=4)
    b.text(1.05, 1.025, '$n$ (records)', transform=b.transAxes, fontsize=7.2, ha='left')
    order = ['country', 'city', 'building_id', 'unit_group', 'within_unit']
    v = variance[variance.weighting.eq('unit_weight') & variance.griffiths_coefficient.eq(0.5)].set_index('component').loc[order]
    lim = sens[sens.weighting.eq('unit_weight') & sens.griffiths_coefficient.eq(0.5)].groupby('component').share.agg(['min', 'max']).loc[order] * 100
    shares = v.share.to_numpy() * 100
    cc.barh(range(5), shares, color=['#68A7B0', '#9DB6D2', '#B09FC5', '#8A365E', '#DBB562'], height=0.65, zorder=2)
    for i, val in enumerate(shares):
        cc.plot([lim.iloc[i]['min'], lim.iloc[i]['max']], [i, i], c='#232323', lw=0.8, zorder=4)
        cc.plot([lim.iloc[i]['min'], lim.iloc[i]['max']], [i, i], ls='', marker='|', ms=5, mew=0.8, c='#232323', zorder=4)
        cc.text(max(val, lim.iloc[i]['max']) + 2, i, f'{val:.1f}', va='center', fontsize=7.8)
    cc.set(yticks=range(5), yticklabels=['Country', 'City', 'Building', 'Within\nbuilding', 'Repeated\nrecords'], ylim=(4.65, -0.65), xlim=(0, 78), xlabel='Variance share (%)')
    cc.set_xticks([0, 20, 40, 60])
    cc.tick_params(axis='y', length=0)
    rr = rooms[rooms.minutes.eq(30)].set_index('dataset').loc[['NUS', 'ENTH']]
    left = np.zeros(2)
    for field, paint, label in [('Same', COLORS['teal'], 'Aligned'), ('Mixed', '#B1A1CA', 'Mixed'), ('Opposed', COLORS['coral'], 'Opposite')]:
        width = rr[field].to_numpy() / rr.windows.to_numpy() * 100
        d.barh([0, 1], width, left=left, height=0.55, color=paint, ec='white', lw=0.4, zorder=2)
        for i, w in enumerate(width):
            if w < 12:
                d.annotate(f'{w:.1f}%', (left[i] + w / 2, i - 0.275), xytext=(98.5, i - 0.46), ha='right', va='center', fontsize=6.7, arrowprops={'arrowstyle': '-', 'lw': 0.45, 'color': '#6F6261'}, zorder=4)
            else:
                d.text(left[i] + w / 2, i, f'{w:.1f}%', ha='center', va='center', fontsize=7.2, color='white' if field == 'Same' else '#28262A', zorder=3)
        left += width
    d.set(yticks=[0, 1], yticklabels=[f"{dict(NUS='Singapore', ENTH='Longitudinal')[name]}\n($n = {int(row.windows)}$)" for name, row in rr.iterrows()], ylim=(1.6, -0.85), xlim=(0, 100), xlabel='Room-time windows (%)')
    d.set_xticks([0, 25, 50, 75, 100])
    d.tick_params(axis='y', length=0)
    f.legend([Patch(fc=c) for c in [COLORS['teal'], '#B1A1CA', COLORS['coral']]], ['Aligned', 'Mixed', 'Opposite'], bbox_to_anchor=(0.776639453, 0.196), loc='center', ncol=3, columnspacing=0.7, fontsize=7)
    s.table('Figure_01_countries.csv', countries)
    s.table('Figure_01_map_points.csv', points)
    s.table('Figure_01_variance.csv', pd.DataFrame({'component': order, 'share_pct': shares, 'min_pct': lim['min'].values, 'max_pct': lim['max'].values}))
    s.table('Figure_01_rooms.csv', rr.reset_index())
    f._hem_meta.update(observations=33845, map_locations=49, countries=15, weather_contexts=9, uncertainty='leave-one-source ranges; not confidence intervals')
    return f

main_FIGURES = {1: main_figure_one}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(1, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Figure 2


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
import pandas as pd
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

main_DISPLAY = {'Malmo': 'Malmö', 'Brasilia': 'Brasília'}

main_COMPOSITIONS = [f'{source} {letter}' for source in ['Multinational', 'China'] for letter in 'ABC']

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_bubble_legend_check(study, name, legend, values, area_scale):
    """Validate legend areas using the same points-squared scale as the data."""
    observed = [float(handle.get_sizes()[0]) for handle in legend.legend_handles[:len(values)]]
    study.check(name + '_legend_areas', np.allclose(observed, np.asarray(values) * area_scale), {'values': values, 'areas_pt2': observed, 'area_scale': area_scale})
    study.check(name + '_legend_marker_scale', legend.markerscale == 1.0, legend.markerscale)

def main_marker_handle(color, marker='o', label=None, size=4.6):
    return Line2D([], [], color=color, marker=marker, ls='', ms=size, label=label, markeredgecolor='white', markeredgewidth=0.45)

def main_heatmap(ax, matrix, cmap, vmin, vmax, fmt, climates):
    im = ax.pcolormesh(np.arange(-0.5, 6), np.arange(-0.5, 9), np.asarray(matrix), cmap=cmap, vmin=vmin, vmax=vmax, shading='flat', edgecolors='white', linewidth=0.5, rasterized=False)
    ax.set_xlim(-0.5, 5.5)
    ax.set_ylim(8.5, -0.5)
    ax.set_yticks(range(9), [main_DISPLAY.get(x, x) for x in climates])
    ax.set_xticks(range(6), range(1, 7))
    ax.set_xticks(np.arange(-0.5, 6), minor=True)
    ax.set_yticks(np.arange(-0.5, 9), minor=True)
    ax.grid(which='minor', color='white', lw=0.6)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='minor', length=0)
    ax.tick_params(which='major', length=0)
    ax.set_xlabel('Demand composition')
    for (i, j), val in np.ndenumerate(np.asarray(matrix)):
        label = '0' if abs(val) < 0.005 else format(val, fmt)
        ax.text(j, i, label, ha='center', va='center', fontsize=7.0, color='white' if val > vmin + 0.58 * (vmax - vmin) else '#3C3448')
    return im

def main_figure_two(s: main_Study):
    counts = s.csv('NOMINAL_QUALIFICATION_COUNTS.csv')
    choices = s.csv('NOMINAL_POOLED_CHOICES_WITH_ORIGINS.csv')
    uncertainty = s.csv('SETTING_SELECTION_STABILITY_ALL_BRANCHES.csv')
    climates = s.climates()
    data = counts.merge(choices, on=['climate', 'composition'], validate='one_to_one', suffixes=('', '_choice'))
    s.check('fig2_pairs', len(data) == 54, len(data))
    s.check('fig2_choice_difference_count', int(data.choices_differ.sum()) == 24, 24)
    s.check('fig2_energy_agreement', np.allclose(data.energy_difference_kwh, data.extra_energy_kwh, atol=1e-08, rtol=0))
    s.check('fig2_qualifying_set_count', int(data.qualified_operation_sets_differ.sum()) == 54, 54)
    data['row'] = data.climate.map({c: i for i, c in enumerate(climates)})
    data['column'] = data.composition.map({c: i for i, c in enumerate(main_COMPOSITIONS)})
    f = main_page(2)
    a = main_axes(f, [0.12, 0.565, 0.315, 0.36], 'a', 'Full-demand limit exceedance', None, letter_x=-0.34)
    b = main_axes(f, [0.635, 0.565, 0.325, 0.36], 'b', 'Energy of the choices', 'x', letter_x=-0.26)
    c = main_axes(f, [0.12, 0.1, 0.34, 0.36], 'c', 'Service shifts with selection', 'both', letter_x=-0.3)
    d = main_axes(f, [0.635, 0.1, 0.27, 0.36], 'd', 'Choices under demand error', None, letter_x=-0.37)
    cm = LinearSegmentedColormap.from_list('excess', ['#F7F4F8', '#DEC5DF', '#B277B8', '#713775'])
    matrix = data.pivot(index='climate', columns='composition', values='zone_mean_selected_full_service_excess_c').loc[climates, main_COMPOSITIONS]
    s.check('fig2_excess_bounds', matrix.to_numpy().max() < 1.0, float(matrix.to_numpy().max()))
    im = main_heatmap(a, matrix, cm, 0, 1, '.2f', climates)
    cax = f.add_axes([0.452, 0.565, 0.013, 0.36])
    cb = f.colorbar(im, cax=cax, ticks=[0, 0.25, 0.5, 0.75, 1])
    cb.set_label('Excess mismatch (°C)', labelpad=4)
    cb.outline.set_linewidth(0.5)
    for i in range(9):
        if i % 2 == 1:
            b.axhspan(i - 0.5, i + 0.5, fc='#F3F4F6', zorder=-2)
    for row in data.itertuples():
        b.scatter(row.energy_difference_kwh / 1000, row.row + (row.column - 2.5) * 0.055, s=43, c=COLORS['plum'] if row.choices_differ else '#C5C7C9', ec='#5D4663' if row.choices_differ else '#82878C', lw=0.45, alpha=0.9, zorder=3)
    b.axvline(0, c='#676C72', ls='--', lw=0.7)
    b.set(yticks=range(9), yticklabels=[main_DISPLAY.get(x, x) for x in climates], ylim=(8.5, -0.5), xlim=(-3, 60), xlabel='Full minus mean choice\n(MWh yr$^{-1}$)')
    b.set_xticks([0, 15, 30, 45, 60])
    b.tick_params(axis='y', length=0)
    b.legend([main_marker_handle('#C5C7C9'), main_marker_handle(COLORS['plum'])], ['Same choice (30)', 'Different choice (24)'], loc='upper right', fontsize=7, handlelength=0.8, borderaxespad=0.4)
    changed = data[data.choices_differ]
    energies = changed.extra_energy_kwh / 1000
    c.scatter(changed.zone_mean_selected_population_loss_vs_full_choice_c, changed.zone_mean_selected_fixed_tail_loss_vs_full_choice_c, s=3.4 * energies, c=COLORS['plum'], alpha=0.62, ec='#8A508C', lw=0.65, zorder=3)
    c.scatter([0], [0], s=25, c='#BFC1C4', ec='white', lw=0.4, zorder=4)
    c.axvline(0, c='#72767C', ls='--', lw=0.65)
    c.axhline(0, c='#72767C', ls='--', lw=0.65)
    c.set(xlim=(-0.08, 1.05), ylim=(-1.55, 1.65), xlabel='Population mismatch,\nmean minus full choice (°C)', ylabel='Least-served mismatch,\nmean minus full choice (°C)')
    c.set_xticks([0, 0.25, 0.5, 0.75, 1])
    c.set_yticks([-1.5, -1, -0.5, 0, 0.5, 1])
    handles = [c.scatter([], [], s=3.4 * e, c=COLORS['plum'], alpha=0.62, ec='#8A508C', lw=0.5) for e in [10, 30, 50]]
    size_legend = c.legend(handles, ['10', '30', '50'], title='Extra energy (MWh yr$^{-1}$)', title_fontsize=7, ncol=3, loc='upper right', fontsize=9, handlelength=2, handletextpad=0.6, columnspacing=0.8, borderaxespad=0.3, markerscale=1)
    main_bubble_legend_check(s, 'fig2_extra_energy', size_legend, [10, 30, 50], 3.4)
    selected = uncertainty[uncertainty.library_and_scenario_design.eq('pooled_study_shared') & uncertainty.scenario.eq('studyGroupSD0.5_personSD0.5')]
    matrix2 = selected.pivot(index='climate', columns='composition', values='conditional_choice_disagreement_fraction').loc[climates, main_COMPOSITIONS] * 100
    s.check('fig2_uncertainty_complete', not matrix2.isna().any().any(), matrix2.shape)
    cm2 = LinearSegmentedColormap.from_list('choice_frequency', ['#F6F8F6', '#B6D3CC', '#52958B', '#17665F'])
    im2 = main_heatmap(d, matrix2, cm2, 0, 100, '.0f', climates)
    cax2 = f.add_axes([0.923, 0.1, 0.013, 0.36])
    cb2 = f.colorbar(im2, cax=cax2, ticks=[0, 25, 50, 75, 100])
    cb2.set_label('Changed choices (%)', fontsize=8, labelpad=3)
    cb2.outline.set_linewidth(0.5)
    s.table('Figure_02_choices.csv', data)
    s.table('Figure_02_uncertainty.csv', selected)
    f._hem_meta.update(settings=54, changed_choices=24, changed_qualifying_sets=54, heatmap_excess_range=[0, 1], uncertainty_scenario='studyGroupSD0.5_personSD0.5')
    return f

main_FIGURES = {2: main_figure_two}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(2, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Figure 3


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import pandas as pd
from matplotlib.ticker import MaxNLocator
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_panel_heading(fig, x, y, letter, title):
    heading = fig.text(x, y, f"({letter.strip('()')}) {title}", fontweight='bold', fontsize=STYLE['title_size'], va='center', gid='panel-title')
    fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [heading]

def main_native_floor_panels(s: main_Study):
    profiles = s.csv('CONSTRAINED_PLACEMENT_PROFILES.csv')
    rep = profiles[profiles.context.eq('Miami') & profiles.composition_id.eq(2026102001)]
    model = s.json('Miami.geometry.json')
    polygons = {z: np.asarray(v, dtype=float) for z, v in model['floors'].items() if z in set(rep.zone_id)}
    s.check('fig3_native_zones', len(polygons) == 15, len(polygons))
    levels = sorted({float(v[:, 2].mean()) for v in polygons.values()})
    slots = {}
    initial = rep[rep.layout.eq('within_floor_relocation_0pct')]
    for zone, v in sorted(polygons.items()):
        starts = v[:, :2]
        edges = np.roll(starts, -1, axis=0) - starts
        boundary = MplPath(np.vstack([starts, starts[:1]]), closed=True)
        n = int(initial.zone_id.eq(zone).sum())
        gx, gy = np.meshgrid(np.arange(starts[:, 0].min(), starts[:, 0].max(), 0.55), np.arange(starts[:, 1].min(), starts[:, 1].max(), 0.55))
        xy = np.column_stack([gx.ravel(), gy.ravel()])
        t = np.clip(((xy[:, None, :] - starts) * edges).sum(axis=2) / (edges * edges).sum(axis=1), 0, 1)
        dist = np.linalg.norm(xy[:, None, :] - starts - t[:, :, None] * edges, axis=2)
        candidates = xy[boundary.contains_points(xy) & (dist.min(axis=1) > 1.1)]
        chosen = []
        nearest = np.full(len(candidates), np.inf)
        idx = int(np.argmin(np.linalg.norm(candidates - candidates.mean(axis=0), axis=1)))
        for _ in range(n):
            chosen.append(candidates[idx])
            nearest = np.minimum(nearest, np.linalg.norm(candidates - candidates[idx], axis=1))
            idx = int(np.argmax(nearest))
        slots[zone] = np.asarray(chosen)
    norm = Normalize(np.floor(rep.centre_c.min()), np.ceil(rep.centre_c.max()))
    out = []
    for ri, layout in enumerate(['within_floor_relocation_0pct', 'within_floor_relocation_25pct']):
        group = rep[rep.layout.eq(layout)]
        s.check(f'fig3_native_layout_{ri}', len(group) == 267, len(group))
        for fi, lev in enumerate(levels):
            panel = {'row': ri, 'floor': fi, 'polygons': [], 'points': [], 'values': []}
            for zone, v in polygons.items():
                if float(v[:, 2].mean()) != lev:
                    continue
                local = group[group.zone_id.eq(zone)].sort_values('profile_id')
                if len(local) != len(slots[zone]):
                    raise ValueError('Rearrangement changed zone capacity')
                panel['polygons'].append(v[:, :2])
                panel['points'].append(slots[zone])
                panel['values'].extend(local.centre_c.tolist())
            panel['points'] = np.vstack(panel['points'])
            panel['values'] = np.asarray(panel['values'])
            out.append(panel)
    return (out, (norm.vmin, norm.vmax), {'mode': 'native_CSV_and_geometry_JSON', 'coordinates': 'schematic slots within real zone polygons'})

def main_figure_three(s: main_Study):
    effects = s.csv('PAIRED_WITHIN_FLOOR_PLACEMENT_EFFECTS.csv')
    frontier = s.csv('DESCRIPTIVE_BUDGET_SUMMARY.csv')
    if s.has('CONSTRAINED_PLACEMENT_PROFILES.csv') and s.has('Miami.geometry.json'):
        panels, bounds, provenance = main_native_floor_panels(s)
    elif s.strict_raw:
        raise FileNotFoundError('Figure 3a needs CONSTRAINED_PLACEMENT_PROFILES.csv and Miami.geometry.json')
    else:
        raise FileNotFoundError('Native profile and geometry inputs are required')
        main_LOG.warning('Figure 3a: replaying source SVG geometry/markers; raw profiles and geometry are absent.')
    f = main_page(3)
    f._hem_meta['panel_a'] = provenance
    main_panel_heading(f, 0.095, 0.963, 'a', 'Office-floor demand placement')
    cmap = LinearSegmentedColormap.from_list('placement', ['#2468A3', '#52A9CB', '#F3DFA2', '#E58E6E', '#B73555'])
    norm = Normalize(*bounds)
    cbax = f.add_axes([0.92, 0.405, 0.014, 0.51])
    cb = f.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbax)
    cb.set_ticks([18, 22, 26, 30])
    cb.ax.tick_params(pad=2, length=2)
    cb.set_label('Demand center (°C)', labelpad=5)
    cb.outline.set_linewidth(0.5)
    columns = [0.115, 0.47]
    rows = [0.755, 0.58, 0.405]
    for ri, x in enumerate(columns):
        f.text(x + 0.2, 0.935, ['Random mixing', '25% move allowance'][ri], ha='center', va='center')
    for fi, y in enumerate(rows):
        f.add_artist(Rectangle((0.045, y), 0.026, 0.16, transform=f.transFigure, fc='#ECF0F3', ec='none', zorder=-2))
        f.text(0.058, y + 0.08, ['Lower floor', 'Middle floor', 'Upper floor'][fi], rotation=90, ha='center', va='center')
    for panel in panels:
        ri, fi = (panel['row'], panel['floor'])
        ax = f.add_axes([columns[ri], rows[fi], 0.4, 0.17])
        polygons = panel['polygons']
        dots = panel['points']
        allxy = np.vstack(polygons)
        lo = allxy.min(axis=0)
        hi = allxy.max(axis=0)
        base = Rectangle(lo, *hi - lo, fc='#FBFAF7', ec='#7F8A92', lw=0.8, zorder=0)
        base.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(1.5, -1.5), alpha=0.16), pe.Normal()])
        ax.add_patch(base)
        for poly in polygons:
            ax.add_patch(Polygon(poly, closed=True, fc='#EDF1F2', alpha=0.48, ec='#9CA8AD', lw=0.55, ls=(0, (3, 2)), zorder=1))
        points = ax.scatter(*dots.T, c=cmap(norm(panel['values'])), s=24, ec='white', lw=0.6, zorder=3)
        points.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.4, -0.4), alpha=0.2), pe.Normal()])
        ax.set_xlim(lo[0] - 1, hi[0] + 1)
        ax.set_ylim(lo[1] - 1, hi[1] + 1)
        ax.set_aspect('equal')
        ax.axis('off')
        if ri == 1 and fi == 2:
            ax.plot([hi[0] - 13, hi[0] - 3], [lo[1] - 2.4, lo[1] - 2.4], lw=1.5, color='black', clip_on=False)
            ax.text(hi[0] - 8, lo[1] - 3.8, '10 m', ha='center', va='top', clip_on=False)
    b = main_axes(f, [0.095, 0.07, 0.235, 0.255], 'b', 'Population service gain', letter_x=-0.3)
    c = main_axes(f, [0.425, 0.07, 0.235, 0.255], 'c', 'Least-served service gain', letter_x=-0.25)
    d = main_axes(f, [0.755, 0.07, 0.205, 0.255], 'd', 'Service requirements', letter_x=-0.28)
    all_stats = []
    for ax, field in [(b, 'population_gain_c'), (c, 'fixed_tail_gain_c')]:
        for control, paint, marker, label in [('personal_center', COLORS['blue'], 'o', 'Personal center'), ('personal_range', COLORS['plum'], 's', 'Personal range')]:
            stats = effects[effects.control.eq(control)].groupby('maximum_relocation_fraction')[field].agg(median='median', q25=lambda v: v.quantile(0.25), q75=lambda v: v.quantile(0.75))
            s.check(f'fig3_{field}_{control}_groups', len(stats) == 3, len(stats))
            x = np.r_[0, stats.index.to_numpy() * 100]
            med, low, high = (np.r_[0, stats[k].to_numpy()] for k in ['median', 'q25', 'q75'])
            ax.fill_between(x, low, high, color=paint, alpha=0.15, lw=0)
            ax.plot(x, med, c=paint, marker=marker, ms=4.8, lw=1.5, label=label, zorder=3)
            for xx, m, lower, h in zip(x, med, low, high):
                all_stats.append({'metric': field, 'control': control, 'move_allowance_pct': xx, 'median': m, 'q25': lower, 'q75': h})
        ax.set(xlim=(-4, 104), xticks=[0, 25, 50, 100], xlabel='Move allowance (%)', ylabel='Mismatch reduction (°C)')
        ax.yaxis.set_major_locator(MaxNLocator(5))
        ax.set_ylim(bottom=-0.02)
    b.set_ylim(-0.02, 0.56)
    c.set_ylim(-0.04, 1.43)
    f.legend(*b.get_legend_handles_labels(), loc='center', bbox_to_anchor=(0.38, 0.375), ncol=2, handlelength=1.4)
    for fraction, paint, linestyle, label in [(1.0, '#454B51', '--', 'Reference'), (0.9, '#38876F', '-', '90% reference')]:
        g = frontier[frontier.arm.eq('vav_strong_mixed_library') & frontier.budget_type.eq('reference_relative') & frontier.reference_fraction.eq(fraction)].sort_values('maximum_relocation_fraction')
        s.check(f'fig3_frontier_{fraction}', len(g) == 4, g.feasible_settings.tolist())
        d.plot(g.maximum_relocation_fraction * 100, g.feasible_settings, c=paint, ls=linestyle, marker='o', ms=3.7, lw=1.3, label=label)
        if fraction == 0.9:
            for xx, yy in zip(g.maximum_relocation_fraction * 100, g.feasible_settings):
                d.annotate(str(int(yy)), (xx, yy), xytext=(0, -12 if yy > 50 else 7), textcoords='offset points', ha='center', fontsize=6.9, color=paint)
    d.set(xlim=(-4, 108), ylim=(20, 60), xticks=[0, 25, 50, 100], yticks=[20, 30, 40, 50, 60], xlabel='Move allowance (%)', ylabel='Feasible settings')
    d.legend(loc='lower right', handlelength=1.2)
    s.table('Figure_03_service_curves.csv', pd.DataFrame(all_stats))
    s.table('Figure_03_frontiers.csv', frontier)
    f._hem_meta.update(profile_count_per_layout=267, move_allowance=0.25, maximum_moves=66, uncertainty='interquartile ranges; not confidence intervals', furniture_added=False)
    return f

main_FIGURES = {3: main_figure_three}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(3, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Figure 4


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_marker_handle(color, marker='o', label=None, size=4.6):
    return Line2D([], [], color=color, marker=marker, ls='', ms=size, label=label, markeredgecolor='white', markeredgewidth=0.45)

def main_figure_four(s: main_Study):
    hp = s.csv('independent_pthp_MATCHED_CROSS_SYSTEM_EFFECTS.csv')
    reset = s.csv('vav_coordinated_outdoor_reset_MATCHED_CROSS_SYSTEM_EFFECTS.csv')
    decomp = s.csv('DECOMPOSITION.csv')
    hp = hp[hp.layout.eq('within_floor_relocation_25pct')]
    reset = reset[reset.layout.eq('within_floor_relocation_25pct')]
    keys = ['climate', 'composition_id', 'control', 'layout']
    common = hp[keys].merge(reset, on=keys, validate='one_to_one')
    s.check('fig4_pair_support', len(hp) == 105 and len(reset) == 108 and (len(common) == 105), [len(hp), len(reset), len(common)])
    f = main_page(4)
    a = main_axes(f, [0.12, 0.565, 0.34, 0.36], 'a', 'HVAC changes energy effects', letter_x=-0.27)
    b = main_axes(f, [0.63, 0.565, 0.34, 0.36], 'b', 'Effect of supply-air reset', letter_x=-0.28)
    c = main_axes(f, [0.12, 0.115, 0.34, 0.35], 'c', 'End uses behind the change', 'x', letter_x=-0.27)
    d = main_axes(f, [0.63, 0.115, 0.34, 0.35], 'd', 'Sources of service gain', 'y', letter_x=-0.28)
    for ax, frame in [(a, hp), (b, reset)]:
        ax.axhline(0, c='#676D74', lw=0.7, ls='--')
        ax.axvline(0, c='#676D74', lw=0.7, ls='--')
        for control, paint, marker, label in [('personal_center', COLORS['blue'], 'o', 'Personal center'), ('personal_range', COLORS['plum'], 's', 'Personal range')]:
            g = frame[frame.control.eq(control)]
            ax.scatter(g.energy_change_pct_vav, g.energy_change_pct_alternative, c=paint, marker=marker, s=39, alpha=0.8, ec='white', lw=0.4, label=label, zorder=3)
        ax.set_xlim(-2.5, 13)
        ax.set_xticks([0, 5, 10])
        ax.set_xlabel('Original VAV energy change (%)')
    f.legend(*a.get_legend_handles_labels(), loc='center', bbox_to_anchor=(0.545, 0.505), ncol=2)
    a.add_patch(Rectangle((0, -3), 13, 3, fc='#DDE9F0', alpha=0.45, zorder=-2))
    a.set(ylim=(-3, 3), yticks=[-3, -2, -1, 0, 1, 2, 3], ylabel='Zone heat-pump\nenergy change (%)')
    a.text(0.97, 0.035, 'VAV increases;\nheat pumps decrease', transform=a.transAxes, ha='right', va='bottom', fontsize=7.5, color='#626B74', fontstyle='italic')
    b.plot([-2.5, 13], [-2.5, 13], c='#4C5157', ls='--', lw=0.8, zorder=1)
    b.set(ylim=(-2.5, 13), yticks=[0, 5, 10], xlabel='Original air supply:\nenergy change (%)', ylabel='After supply-air reset:\nenergy change (%)')
    b.text(0.85, 0.96, 'Equal\nenergy effects', ha='right', va='top', transform=b.transAxes, fontsize=7)
    carriers = [('heating_electric_change_mwh', 'Electric heating', '#B65F69'), ('cooling_electric_change_mwh', 'Cooling', '#648DB4'), ('fans_electric_change_mwh', 'Fans', '#90AA90'), ('facility_gas_hhv_change_mwh', 'Gas', '#CEB27B')]
    systems = [(hp, 'vav', 'Original VAV'), (common, 'alternative', 'VAV reset'), (hp, 'alternative', 'Zone heat pumps')]
    part_rows = []
    for y, (frame, suffix, label) in enumerate(systems):
        total = frame['facility_electric_change_mwh_' + suffix] + frame['facility_gas_hhv_change_mwh_' + suffix]
        parts = frame[[field + '_' + suffix for field, _, _ in carriers]].sum(axis=1)
        s.check(f'fig4_energy_closure_{label}', np.allclose(parts, total, atol=1e-06, rtol=1e-08))
        positive = negative = 0.0
        for field, component, paint in carriers:
            val = float(frame[field + '_' + suffix].mean())
            start = positive if val >= 0 else negative
            c.barh(y, val, left=start, color=paint, height=0.5, ec='white', lw=0.4, zorder=2)
            if abs(val) >= 1.25:
                c.text(start + val / 2, y, f'{val:.1f}', ha='center', va='center', color='white', fontsize=7.2, zorder=4)
            if val >= 0:
                positive += val
            else:
                negative += val
            part_rows.append({'system': label, 'component': component, 'mean_change_mwh': val})
        value = float(total.mean())
        c.scatter(value, y, c='black', s=34, ec='white', lw=0.6, zorder=5)
        tx = max(value, positive) + 0.7
        c.text(tx, y, f'{value:+.2f}', va='center', ha='left', fontsize=8, fontweight='bold', zorder=5)
    c.axvline(0, c='#60676F', lw=0.7, ls='--')
    c.set(yticks=range(3), yticklabels=['Original\nVAV', 'VAV\nreset', 'Zone heat\npumps'], ylim=(3.05, -0.45), xlim=(-3.5, 24), xlabel='Mean annual energy change (MWh)')
    c.set_xticks([0, 5, 10, 15, 20])
    c.tick_params(axis='y', length=0)
    c.legend([Patch(fc=paint) for _, _, paint in carriers] + [main_marker_handle('black')], [label for _, label, _ in carriers] + ['Total'], loc='lower right', ncol=2, fontsize=7, columnspacing=0.9, handlelength=0.9)
    dec_rows = []
    for x, system in enumerate(['vav', 'pthp']):
        local = decomp[decomp.system.eq(system)]
        s.check(f'fig4_service_support_{system}', len(local) == 105, len(local))
        bottom = 0.0
        for field, paint, label in [('population_assignment_step_c', COLORS['blue'], 'Assignment'), ('population_trajectory_step_c', COLORS['plum'], 'Temperature')]:
            val = float(local[field].mean())
            d.bar(x, val, bottom=bottom, width=0.55, color=paint, ec='white', lw=0.4, zorder=2)
            d.text(x, bottom + val / 2, f'{val:.3f}', ha='center', va='center', color='white', fontsize=8)
            bottom += val
            dec_rows.append({'system': system, 'step': label, 'mean_gain_c': val})
        s.check(f'fig4_service_closure_{system}', abs(bottom - local.population_total_step_c.mean()) < 1e-10)
        d.text(x, bottom + 0.012, f'{bottom:.3f}', ha='center', fontsize=9, fontweight='bold')
    d.set(xticks=[0, 1], xticklabels=['VAV', 'Heat pumps'], ylim=(0, 0.4), yticks=[0, 0.1, 0.2, 0.3, 0.4], ylabel='Population mismatch\nreduction (°C)')
    d.legend([Patch(fc=COLORS['blue']), Patch(fc=COLORS['plum'])], ['Assignment step', 'Temperature step'], loc='upper left', fontsize=7, borderaxespad=0.3, handlelength=0.9)
    s.table('Figure_04_heatpump_pairs.csv', hp)
    s.table('Figure_04_reset_pairs.csv', reset)
    s.table('Figure_04_energy_components.csv', pd.DataFrame(part_rows))
    s.table('Figure_04_service_decomposition.csv', pd.DataFrame(dec_rows))
    f._hem_meta.update(common_pairs=105, reset_pairs=108, comparison='arranged minus mixed, within each system', decomposition='order-dependent accounting steps; not causal effects')
    return f

main_FIGURES = {4: main_figure_four}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(4, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Figure 5


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pandas as pd
from matplotlib.ticker import MaxNLocator
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

main_DISPLAY = {'Malmo': 'Malmö', 'Brasilia': 'Brasília'}

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_marker_handle(color, marker='o', label=None, size=4.6):
    return Line2D([], [], color=color, marker=marker, ls='', ms=size, label=label, markeredgecolor='white', markeredgewidth=0.45)

def main_figure_five(s: main_Study):
    pairs = s.csv('PAIRED_EVENT_INFORMATION_REVIEW.csv')
    scores = s.csv('FULL_REPLAY_ACTION_SCORES.csv')
    draws = s.csv('PAIRED_FROZEN_POWER_DRAWS.csv')
    pairs = pairs[pairs.control.eq('building_search') & pairs.normal_energy_reference_factor.eq(1.0) & pairs.policy.eq('maximum_power')].copy()
    s.check('fig5_all_events', len(pairs) == 432 and pairs.normal_eligible_engineering_resolved_support.all(), len(pairs))
    pairs['checked_mean_kw'] = np.where(pairs.mean_selected_full_service_qualified.eq(True), pairs.mean_available_power_kw, 0.0)
    climates = s.climates()
    fields = ['mean_available_power_kw', 'checked_mean_kw', 'full_available_power_kw']
    summary = pairs.groupby('climate')[fields].mean().loc[climates]
    f = main_page(5)
    a = main_axes(f, [0.235, 0.535, 0.725, 0.39], 'a', 'Individual demands change response estimates', 'x', letter_x=-0.29)
    a._left_title.set_x((0.09 - 0.235) / 0.725)
    b = main_axes(f, [0.09, 0.105, 0.24, 0.33], 'b', 'Service behind\npower estimates', letter_x=-0.3)
    c = main_axes(f, [0.435, 0.105, 0.2, 0.33], 'c', 'Service across\nresponse windows', 'x', letter_x=-0.39)
    d = main_axes(f, [0.755, 0.105, 0.205, 0.33], 'd', 'Demand error and\nusable power', letter_x=-0.4)
    for i, row in enumerate(summary.itertuples()):
        if i % 2 == 1:
            a.axhspan(i - 0.5, i + 0.5, fc='#F5F6F7', zorder=-2)
        a.plot([row.checked_mean_kw, row.mean_available_power_kw], [i, i], c='#969DA4', lw=0.8, zorder=1)
    a.scatter(summary.mean_available_power_kw, range(9), marker='o', s=90, c=COLORS['gold'], ec='#4F4840', lw=0.6, zorder=4)
    a.scatter(summary.checked_mean_kw, np.arange(9) - 0.16, marker='s', s=60, c=COLORS['coral'], ec='#595052', lw=0.6, zorder=5)
    a.scatter(summary.full_available_power_kw, np.arange(9) + 0.16, marker='D', s=64, c=COLORS['teal'], ec=COLORS['teal'], lw=1.2, zorder=6)
    a.set(yticks=range(9), yticklabels=[main_DISPLAY.get(x, x) for x in climates], ylim=(8.55, -0.6), xlim=(-0.7, 29), xticks=[0, 6, 12, 18, 24], xlabel='Mean event-power reduction (kW)')
    a.tick_params(axis='y', length=0, labelleft=False)
    geography = s.json('ne_110m_observed_countries.json')['features']
    land = s.json('ne_110m_land.geojson')['features']
    context = {r['climate']: r for r in s.json('GLOBAL_CLIMATES.json')['climates']}
    country_keys = {'United States': 'usa', 'Canada': 'canada', 'Sweden': 'sweden', 'India': 'india', 'China': 'china', 'Tunisia': 'tunisia', 'Australia': 'australia', 'Brazil': 'brazil'}
    for i, climate in enumerate(climates):
        row = context[climate]
        yp = 0.535 + 0.39 * (8.55 - i) / 9.15
        f.text(0.137, yp, main_DISPLAY.get(climate, climate), ha='right', va='center')
        mini = f.add_axes([0.155, yp - 0.018, 0.055, 0.036])
        rings = []
        for feature in geography:
            if feature['country_key'] == country_keys.get(row['country']):
                rings = feature['rings']
                break
        if rings:
            rings = sorted(rings, key=lambda r: np.ptp(np.asarray(r)[:, 0]) * np.ptp(np.asarray(r)[:, 1]), reverse=True)[:1]
            for ring in rings:
                mini.add_patch(Polygon(ring, fc='#B9BDC2', ec='none'))
            xy = np.vstack(rings)
            lo = xy.min(axis=0)
            hi = xy.max(axis=0)
            mini.set_xlim(lo[0] - 1, hi[0] + 1)
            mini.set_ylim(lo[1] - 1, hi[1] + 1)
        else:
            for feature in land:
                geometry = feature['geometry']
                polys = [geometry['coordinates']] if geometry['type'] == 'Polygon' else geometry['coordinates']
                for poly in polys:
                    mini.add_patch(Polygon(poly[0], fc='#B9BDC2', ec='none'))
            mini.set_xlim(98, 109)
            mini.set_ylim(-3, 8)
        mini.scatter(row['longitude'], row['latitude'], s=5, c=COLORS['teal'], ec='white', lw=0.25)
        mini.set_aspect('equal')
        mini.axis('off')
    a.legend([main_marker_handle(COLORS['gold']), main_marker_handle(COLORS['coral'], 's'), main_marker_handle(COLORS['teal'], 'D')], ['Zone-mean selection and scoring', 'Zone-mean selection, full-demand check', 'Full-demand selection and scoring'], loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=7.1, borderaxespad=0.3)
    selected = pairs[pairs.mean_selected_action.notna()].merge(scores, left_on=['climate', 'composition_id', 'control', 'event_id', 'mean_selected_action'], right_on=['climate', 'composition_id', 'control', 'event_id', 'action_id'], validate='one_to_one')
    s.check('fig5_selected_actions', len(selected) == 423, len(selected))
    excess_fields = ['mean_selected_full_baseline_window_excess_c', 'mean_selected_full_action_precondition_excess_c', 'mean_selected_full_action_event_excess_c', 'mean_selected_full_action_recovery_excess_c']
    selected['display_maximum_service_excess_c'] = selected[excess_fields].max(axis=1)
    b.scatter(selected.mean_available_power_kw, selected.display_maximum_service_excess_c, c=COLORS['plum'], s=20, alpha=0.38, lw=0, zorder=3)
    b.axhline(0, c='#777D83', ls='--', lw=0.65)
    b.set(ylim=(-0.035, 1.02), xlim=(-1, 56), xticks=[0, 20, 40], yticks=[0, 0.3, 0.6, 0.9], xlabel='Event-power\nreduction (kW)', ylabel='Service excess (°C)')
    b.text(0.95, 0.95, '$n = 423$\nselected actions', transform=b.transAxes, ha='right', va='top', fontsize=7.1)
    common = selected[selected.common_baseline_window_support]
    s.check('fig5_common_baseline', len(common) == 376, len(common))
    stage_rows = []
    for j, (metric, paint, label) in enumerate([('population_mean_mismatch_c', COLORS['blue'], 'Population'), ('fixed_tail_mean_mismatch_c', COLORS['coral'], 'Least-served')]):
        vals = []
        for window in ['precondition', 'event', 'recovery']:
            obs = common['action_' + window + '_' + metric]
            ref = common['reference_' + window + '_' + metric]
            evaluated = np.isfinite(obs) & np.isfinite(ref)
            val = int((evaluated & obs.gt(ref + 1e-09)).sum())
            vals.append(val)
            stage_rows.append({'window': window, 'metric': label, 'count': val, 'evaluated': int(evaluated.sum())})
        y = np.arange(3) + (j - 0.5) * 0.36
        c.barh(y, vals, height=0.29, color=paint, zorder=2)
        for yy, val in zip(y, vals):
            c.text(val + 7, yy, str(val), va='center', fontsize=7.6)
    c.set(yticks=range(3), yticklabels=['Before', 'Event', 'Recovery'], ylim=(2.55, -1.35), xlim=(0, 315), xticks=[0, 150, 300], xlabel='Events outside\nservice limits')
    c.tick_params(axis='y', length=0)
    c.legend([Patch(fc=COLORS['blue']), Patch(fc=COLORS['coral'])], ['Population', 'Least-served'], loc='upper left', fontsize=6.9, borderaxespad=0.25, handlelength=0.8)
    local = draws[draws.control.eq('building_search') & draws.budget.eq(1.0) & draws.support.eq('all_normal_available')]
    dr = []
    for prefix, ls, source in [('sourceSD', '-', 'Source-shared'), ('studyGroupSD', '--', 'Study-shared')]:
        for field, paint, name in [('mean_selection_full_checked_power_kw', COLORS['coral'], 'Zone-mean selection, full-demand check'), ('full_selection_full_checked_power_kw', COLORS['teal'], 'Full-demand selection and scoring')]:
            means = []
            lo = []
            hi = []
            for val in [0, 0.25, 0.5]:
                vv = local[local.scenario.eq(f'{prefix}{val:g}_personSD0.5')][field]
                s.check(f'fig5_draws_{prefix}_{field}_{val}', len(vv) == 256, len(vv))
                means.append(vv.mean())
                lo.append(vv.quantile(0.05))
                hi.append(vv.quantile(0.95))
                dr.append({'shared_error': source, 'selection': name, 'sd_c': val, 'mean_kw': vv.mean(), 'q05_kw': vv.quantile(0.05), 'q95_kw': vv.quantile(0.95), 'draws': len(vv)})
            d.fill_between([0, 0.25, 0.5], lo, hi, color=paint, alpha=0.1, lw=0)
            d.plot([0, 0.25, 0.5], means, c=paint, ls=ls, lw=1.2, zorder=3)
    d.set(xlim=(-0.015, 0.515), xticks=[0, 0.25, 0.5], xlabel='Shared error SD (°C)', ylabel='Service-preserving\npower (kW)')
    d.yaxis.set_major_locator(MaxNLocator(4))
    d.set_ylim(0.8, max(d.get_ylim()[1], 3.1))
    d.legend([Line2D([], [], c=COLORS['coral'], lw=1.5), Line2D([], [], c=COLORS['teal'], lw=1.5), Line2D([], [], c='black', ls='-', lw=1), Line2D([], [], c='black', ls='--', lw=1)], ['Zone-mean selection', 'Full-demand selection', 'Source-shared', 'Study-shared'], loc='lower left', ncol=1, labelspacing=0.3, columnspacing=1.2)
    s.table('Figure_05_climate_means.csv', summary.reset_index())
    s.table('Figure_05_selected_actions.csv', selected)
    s.table('Figure_05_stage_counts.csv', pd.DataFrame(stage_rows))
    s.table('Figure_05_uncertainty.csv', pd.DataFrame(dr))
    means = pairs[fields].mean().to_dict()
    s.check('fig5_published_means', np.allclose([means[x] for x in fields], [11.532, 2.494, 2.514], atol=0.001), means)
    f._hem_meta.update(all_events=432, mean_selected=423, common_baseline=376, all_event_means=means, error_bands='5th–95th draw percentiles; not confidence intervals', selection='finite-menu perfect-foresight capability')
    return f

main_FIGURES = {5: main_figure_five}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(5, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Figure 6


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Sequence
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import logging
from typing import Any
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

main_LOG = logging.getLogger('hem_figures')

COLORS = {'blue': '#4778A8', 'plum': '#AF668A', 'teal': '#26898B', 'coral': '#DF7D73', 'gold': '#E6B34B', 'gray': '#9CA3AA', 'grid': '#E2E5E8', 'ink': '#000000', 'cool': '#A3BDD2', 'green': '#8BA88B', 'sand': '#CCB16D'}

STYLE = {'width_in': 7.6, 'height_in': 9.4, 'font_size': FONT_SIZE, 'title_size': PANEL_TITLE_SIZE, 'panel_size': PANEL_TITLE_SIZE, 'font': 'Times New Roman', 'figure6_font': 'Times New Roman', 'dpi': DPI}

def main__jsonable(value):
    if isinstance(value, dict):
        return {str(k): main__jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [main__jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value

class main_Study:

    def __init__(self, data_root, output, strict_raw=False):
        self.root, self.output = (Path(data_root).resolve(), Path(output).resolve())
        self.strict_raw = strict_raw
        self.used, self.checks = ({}, {})
        self.output.mkdir(parents=True, exist_ok=True)
        (self.output / 'plot_data').mkdir(exist_ok=True)

    def path(self, name):
        path = data_path(name)
        self.used[path.relative_to(ROOT).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        return path

    def has(self, name):
        return any((Path(r.get('key', r.get('result', ''))).name == name for r in DATA_FILES))

    def csv(self, name: str) -> pd.DataFrame:
        return pd.read_csv(self.path(name), low_memory=False)

    def json(self, name: str) -> dict:
        return json.loads(self.path(name).read_text(encoding='utf-8-sig'))

    def table(self, name: str, df: pd.DataFrame):
        df.to_csv(self.output / 'plot_data' / name, index=False, float_format='%.12g')

    def check(self, name: str, condition: bool, value: Any=None):
        self.checks[name] = {'passed': bool(condition), 'value': main__jsonable(value)}
        if not condition:
            raise ValueError(f'Data integrity check failed: {name}; value={value}')

    def climates(self) -> list[str]:
        return [x['climate'] for x in self.json('GLOBAL_CLIMATES.json')['climates']]

def main_choose_font(request: str) -> str:
    options = {'sans': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif']}.get(request, [request])
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in options:
        if name in available:
            return name
    raise RuntimeError(f'Font {request!r} is not installed. Use --font sans or --font serif.')

def main_style(number: int):
    mpl.rcdefaults()
    family = main_choose_font(STYLE['figure6_font'] if number == 6 else STYLE['font'])
    plt.rcParams.update({'font.family': family, 'font.size': STYLE['font_size'], 'axes.titlesize': STYLE['title_size'], 'axes.titleweight': 'bold', 'axes.titlepad': 8, 'axes.labelsize': STYLE['font_size'], 'axes.labelpad': 5, 'axes.linewidth': 0.65, 'axes.edgecolor': '#454B51', 'axes.labelcolor': COLORS['ink'], 'xtick.labelsize': STYLE['font_size'], 'ytick.labelsize': STYLE['font_size'], 'xtick.color': COLORS['ink'], 'ytick.color': COLORS['ink'], 'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.55, 'ytick.major.width': 0.55, 'text.color': COLORS['ink'], 'legend.frameon': False, 'legend.fontsize': STYLE['font_size'], 'legend.handlelength': 1.4, 'legend.handletextpad': 0.5, 'legend.labelspacing': 0.45, 'grid.color': COLORS['grid'], 'grid.linestyle': ':', 'grid.linewidth': 0.55, 'axes.axisbelow': True, 'savefig.facecolor': 'white', 'figure.facecolor': 'white', 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.hashsalt': 'hem-data-faithful-20261101', 'path.simplify': False, 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold'})
    return family

def main_page(number: int):
    family = main_style(number)
    f = plt.figure(figsize=(STYLE['width_in'], STYLE['height_in']), facecolor='white')
    f._hem_meta = {'figure': number, 'font': family, 'source_mode': 'CSV numerical results'}
    return f

def main_axes(fig, rect, letter: str, heading: str, grid: str | None='both', letter_x=-0.16):
    a = fig.add_axes(rect)
    text = f"({letter.strip('()')}) {heading}" if heading else ''
    title = a.set_title(text, loc='left', fontsize=STYLE['title_size'], fontweight='bold', pad=8)
    title.set_gid('panel-title')
    if heading:
        fig._hem_panel_headings = getattr(fig, '_hem_panel_headings', []) + [title]
    if grid:
        a.grid(axis=grid)
    return a

def main_bubble_legend_check(study, name, legend, values, area_scale):
    """Validate legend areas using the same points-squared scale as the data."""
    observed = [float(handle.get_sizes()[0]) for handle in legend.legend_handles[:len(values)]]
    study.check(name + '_legend_areas', np.allclose(observed, np.asarray(values) * area_scale), {'values': values, 'areas_pt2': observed, 'area_scale': area_scale})
    study.check(name + '_legend_marker_scale', legend.markerscale == 1.0, legend.markerscale)

def main_native_response_curves(s: main_Study, example: pd.Series):
    plan = s.csv('SINGLE_EVENT_ACTION_PLAN.csv')
    normals = s.csv('NORMAL_OPERATION_REPLAY_COMPLETED.csv').set_index('normal_id')
    actions = s.csv('ACTION_FULL_COMPLETED.csv').set_index('run_id')
    context = plan[plan.climate.eq(example.climate) & plan.composition_id.eq(example.composition_id) & plan.control.eq(example.control) & plan.event_id.eq(example.event_id)]
    curves = []
    for column, net in [('selected_action', 'net_full_window_electricity_reduction_kwh'), ('positive_net_selected_action', 'positive_net_full_window_kwh')]:
        operation = context[context.action_id.eq(example[column])].iloc[0]
        normal_name = Path(str(normals.loc[operation.normal_id, 'hourly_path']).replace('\\', '/')).name
        action_name = Path(str(actions.loc[operation.run_id, 'hourly_path']).replace('\\', '/')).name
        with np.load(s.path(normal_name), allow_pickle=False) as file:
            baseline = dict(file)
        with np.load(s.path(action_name), allow_pickle=False) as file:
            action = dict(file)

        def stamps(a):
            return pd.DatetimeIndex([pd.Timestamp(2001, int(m), int(d)) + pd.Timedelta(hours=int(h) - 1) for m, d, h in zip(a['month'], a['day'], a['hour'])])
        clock = stamps(baseline)
        if not clock.equals(stamps(action)):
            raise ValueError('Normal and action clocks differ')
        event = json.loads(operation.event_json)
        start = pd.Timestamp(event['start_timestamp'])
        begin = start - pd.Timedelta(hours=int(event['precondition_hours']))
        end = start + pd.Timedelta(hours=int(event['duration_hours']) + int(event['recovery_hours']))
        mask = (clock >= begin) & (clock < end)
        delta = baseline['facility_electric_kwh'] - action['facility_electric_kwh']
        if abs(delta[mask].sum() - example[net]) > 1e-08:
            raise ValueError('Native time-series endpoint disagrees with selected action')
        hours = (clock[mask] - start).total_seconds().to_numpy() / 3600
        curves.append((np.r_[hours[0], hours + 1], np.r_[0, np.cumsum(delta[mask])]))
    return (curves, {'mode': 'native_hourly_NPZ', 'no_smoothing': True, 'no_uncertainty_bands': True})

def main_figure_six(s: main_Study):
    choices = s.csv('ONE_ACTION_PER_EVENT_BUDGET_CHOICES.csv')
    scores = s.csv('ISOLATED_EVENT_ACTION_SCORES.csv')
    choices = choices[choices.control.eq('building_search') & choices.normal_energy_reference_factor.eq(1.0) & choices.status.eq('selected')].copy()
    s.check('fig6_selected_events', len(choices) == 112 and choices.positive_net_selected_action.notna().all(), len(choices))
    keys = ['climate', 'composition_id', 'control', 'event_id']
    lookup = scores.set_index(keys + ['action_id'], verify_integrity=True)
    negative = choices[choices.net_full_window_electricity_reduction_kwh.lt(0)].copy()
    negative['distance'] = (negative.net_full_window_electricity_reduction_kwh - negative.net_full_window_electricity_reduction_kwh.median()).abs()
    example = negative.sort_values(['distance', *keys]).iloc[0]
    s.check('fig6_negative_net_events', len(negative) == 44, len(negative))
    npzs = ['normal_f620f82ff7dd4584423772e3.npz', 'isolated_4caeea1e4b24d6220d74d1dc.npz', 'isolated_0e545d46d7ce446398d31866.npz']
    if all((s.has(name) for name in npzs)):
        curves, provenance = main_native_response_curves(s, example)
    elif s.strict_raw:
        raise FileNotFoundError('Figure 6a/b needs the three original hourly NPZ files listed in INPUT_AUDIT.json')
    else:
        raise FileNotFoundError('Native hourly response inputs are required')
        main_LOG.warning('Figure 6a/b: replaying original SVG curve coordinates; hourly NPZ files are absent.')
        for (_, y), column in zip(curves, ['net_full_window_electricity_reduction_kwh', 'positive_net_full_window_kwh']):
            s.check(f'fig6_vector_endpoint_{column}', abs(y[-1] - example[column]) < 2e-05, {'svg': y[-1], 'csv': example[column]})
    f = main_page(6)
    f._hem_meta['panel_ab'] = provenance
    a = main_axes(f, [0.12, 0.565, 0.34, 0.36], '(a)', 'Event savings and recovery', None, letter_x=-0.27)
    b = main_axes(f, [0.63, 0.565, 0.34, 0.36], '(b)', 'First six hours', None, letter_x=-0.28)
    c = main_axes(f, [0.12, 0.115, 0.34, 0.35], '(c)', 'Power and electricity balance', None, letter_x=-0.27)
    d = main_axes(f, [0.63, 0.115, 0.34, 0.35], '(d)', 'Where electricity shifts', None, letter_x=-0.28)
    for ax in [a, b]:
        ax.fill_between(curves[0][0], curves[0][1], curves[1][1], color=COLORS['teal'], alpha=0.12, lw=0, zorder=1)
        ax.axvspan(-2, 0, fc='#DCE6EF', alpha=0.6, lw=0, zorder=-2)
        ax.axvspan(0, 2, fc='#F4E8D5', alpha=0.55, lw=0, zorder=-2)
        ax.axhline(0, c='#767B80', ls='--', lw=0.65, zorder=0)
        for (t, y), paint, label in zip(curves, [COLORS['coral'], COLORS['teal']], ['Maximum event power', 'Positive net electricity']):
            ax.plot(t, y, c=paint, lw=1.65, marker='o', markevery=4, ms=3.8, mec='white', mew=0.35, label=label, zorder=3)
        ax.set(ylim=(-25, 55), xlabel='Hours from event start (h)')
    a.set(xlim=(-3, 52), xticks=[0, 12, 24, 36, 48], yticks=[-20, 0, 20, 40], ylabel='Cumulative electricity\nreduction (kWh)')
    a.add_patch(Rectangle((-2, -23), 6, 64, fill=False, ec='#99A4AD', ls='--', lw=0.6, zorder=2))
    a.text(0.19, 0.8, 'Zoom (b)', ha='left', va='center', transform=a.transAxes, fontsize=8)
    a.legend(loc='upper right', fontsize=7.1, borderaxespad=0.3, handlelength=1.2)
    for (t, y), paint in zip(curves, [COLORS['coral'], COLORS['teal']]):
        mask = (t >= -2) & (t <= 4)
        b.plot(t[mask], y[mask], ls='', marker='o', ms=3.7, c=paint, mec='white', mew=0.35, zorder=4)
    b.set(xlim=(-2, 4), xticks=[-2, -1, 0, 1, 2, 3, 4], yticks=[-20, 0, 20, 40], ylabel='Cumulative electricity\nreduction (kWh)')
    b.text(-1, 52, 'Before\n−2 to 0 h', ha='center', va='top', fontsize=7.0)
    b.text(1, 52, 'Event\n0 to 2 h', ha='center', va='top', fontsize=7.0)
    b.text(3, 52, 'Recovery', ha='center', va='top', fontsize=7.0)
    power = choices.event_power_reduction_kw - choices.positive_net_event_power_kw
    net = choices.positive_net_full_window_kwh - choices.net_full_window_electricity_reduction_kwh
    s.check('fig6_nested_policy_order', power.min() > -1e-09 and net.min() > -1e-09, [power.min(), net.min()])
    coincident = pd.DataFrame({'power_cost_kw': power.round(6), 'net_gain_kwh': net.round(6)}).groupby(['power_cost_kw', 'net_gain_kwh']).size().reset_index(name='cases')
    s.check('fig6_bubble_count', coincident.cases.sum() == 112, int(coincident.cases.sum()))
    c.scatter(coincident.power_cost_kw, coincident.net_gain_kwh, s=5.0 * coincident.cases, c='#AD8CB7', alpha=0.52, ec='#92729C', lw=0.55, zorder=3)
    c.axvline(0, c='#737A80', ls='--', lw=0.65)
    c.axhline(0, c='#737A80', ls='--', lw=0.65)
    c.set(xlim=(-0.23, max(power) * 1.12), ylim=(-1, net.max() * 1.18), xticks=[0, 0.5, 1, 1.5, 2], yticks=[0, 3, 6, 9, 12], xlabel='Event-power cost (kW)', ylabel='Net electricity gained (kWh)')
    c.text(0.96, 0.96, '$n = 112$ events', ha='right', va='top', transform=c.transAxes, fontsize=8)
    hs = [c.scatter([], [], s=5.0 * n, c='#AD8CB7', alpha=0.52, ec='#92729C', lw=0.55) for n in [1, 10, 50]]
    size_legend = c.legend(hs, ['1', '10', '50'], title='Coincident cases', title_fontsize=7.5, loc='lower right', bbox_to_anchor=(0.985, 0.15), ncol=1, fontsize=9, columnspacing=1.1, handlelength=2.2, handletextpad=0.65, markerscale=1)
    main_bubble_legend_check(s, 'fig6_coincident_cases', size_legend, [1, 10, 50], 5.0)
    values = []
    part_rows = []
    for col, policy in [('selected_action', 'Maximum event power'), ('positive_net_selected_action', 'Positive net electricity')]:
        rows = [lookup.loc[tuple((row[k] for k in keys)) + (row[col],)] for row in choices.to_dict('records')]
        frame = pd.DataFrame(rows)
        parts = np.column_stack([-frame.preconditioning_facility_electricity_change_kwh, frame.event_facility_electricity_reduction_kwh, -frame.recovery_facility_electricity_change_kwh])
        s.check(f'fig6_energy_closure_{policy}', np.allclose(parts.sum(axis=1), frame.precondition_event_recovery_facility_net_reduction_kwh, atol=1e-08, rtol=0))
        value = np.r_[parts.mean(axis=0), parts.sum(axis=1).mean()]
        values.append(value)
        for stage, v in zip(['Before', 'Event', 'Recovery', 'Net'], value):
            part_rows.append({'policy': policy, 'stage': stage, 'mean_reduction_kwh': v, 'paired_events': 112})
    for j, (value, paint) in enumerate(zip(values, [COLORS['coral'], COLORS['teal']])):
        yy = np.arange(4) + (j - 0.5) * 0.36
        d.barh(yy, value, height=0.29, color=paint, alpha=0.9, zorder=2)
        for y, v in zip(yy, value):
            d.text(v + (0.6 if v >= 0 else -0.6), y, f'{v:.2f}', ha='left' if v >= 0 else 'right', va='center', fontsize=6.6)
    for v in [0.5, 1.5, 2.5]:
        d.axhline(v, c='#E0E3E5', lw=0.45, zorder=0)
    d.plot([0, 0], [-0.6, 3.4], c='#737A80', ls='--', lw=0.65)
    d.set(yticks=range(4), yticklabels=['Before\n(−2 to 0 h)', 'Event\n(0 to 2 h)', 'Recovery\n(2 to 50 h)', 'Net\n(−2 to 50 h)'], ylim=(4.8, -0.6), xlim=(-23, 29), xticks=[-20, -10, 0, 10, 20], xlabel='Mean electricity\nreduction (kWh)')
    d.tick_params(axis='y', length=0, labelsize=7)
    d.legend([Patch(fc=COLORS['coral']), Patch(fc=COLORS['teal'])], ['Maximum event power', 'Positive net electricity'], loc='lower right', ncol=1, fontsize=7.5, labelspacing=0.3, handlelength=1)
    s.table('Figure_06_coincident_events.csv', coincident)
    s.table('Figure_06_phase_means.csv', pd.DataFrame(part_rows))
    curve_rows = []
    for name, (t, y) in zip(['maximum_power', 'positive_net'], curves):
        for tt, yy in zip(t, y):
            curve_rows.append({'policy': name, 'hour_from_event_start': tt, 'cumulative_electricity_reduction_kwh': yy, 'source_mode': provenance['mode']})
    s.table('Figure_06_example_curves.csv', pd.DataFrame(curve_rows))
    f._hem_meta.update(paired_events=112, negative_net_maxpower_events=44, example={k: str(example[k]) for k in keys}, curve_origin=provenance, electricity_phase_means=part_rows, uncertainty_bands='not present for a single event')
    return f

main_FIGURES = {6: main_figure_six}

def main_export(fig, number: int, s: main_Study, formats: list[str], dpi: int):
    base = s.output / f'Figure_{number:02}'
    for artist in fig.findobj(mpl.collections.Collection):
        artist.set_rasterized(False)
    for ax in fig.axes:
        cb = getattr(ax, '_colorbar', None)
        if cb is not None and cb.solids is not None:
            cb.solids.set_edgecolor('face')
            cb.solids.set_linewidth(0.05)
    for ax in fig.axes:
        for collection in ax.collections:
            if isinstance(collection, mpl.collections.PathCollection) and len(collection.get_offsets()) > 0:
                collection.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.35, -0.35), alpha=0.14), pe.Normal()])
        for patch in ax.patches:
            if isinstance(patch, Rectangle) and patch.get_zorder() >= 1:
                patch.set_path_effects([pe.SimplePatchShadow(shadow_rgbFace=(0.2, 0.24, 0.28), offset=(0.45, -0.45), alpha=0.1), pe.Normal()])
        for line in ax.lines:
            if line.get_linestyle() == '-' and line.get_linewidth() >= 1.2 and (line.get_color() not in ('black', '#333333')):
                line.set_path_effects([pe.Stroke(linewidth=line.get_linewidth() + 0.7, foreground='white', alpha=0.8), pe.Normal()])
    for text in fig.findobj(mpl.text.Text):
        text.set_fontfamily('Times New Roman')
        is_title = text.get_gid() == 'panel-title' or any((text is ax._left_title or text is ax.title for ax in fig.axes))
        text.set_fontsize(10 if is_title and text.get_fontweight() == 'bold' else 9)
        if text.get_color() not in ('white', '#ffffff'):
            text.set_color('black')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig._hem_meta['heading_alignment'] = [{'text': heading.get_text(), 'single_text_artist': True, 'left_fraction': float(heading.get_transform().transform(heading.get_position())[0] / fig.bbox.width)} for heading in fig._hem_panel_headings]
    same_column = {1: [(0, 1), (2, 3)], 2: [(0, 2), (1, 3)], 3: [(0, 1)], 4: [(0, 2), (1, 3)], 5: [(0, 1)], 6: [(0, 2), (1, 3)]}
    anchors = fig._hem_meta['heading_alignment']
    for first, second in same_column[number]:
        assert abs(anchors[first]['left_fraction'] - anchors[second]['left_fraction']) < 1e-06
    if number in (4, 6):
        boxes = [ax.get_position().bounds for ax in fig.axes[:4]]
        assert max((b[2] for b in boxes)) - min((b[2] for b in boxes)) < 1e-12
        assert abs(boxes[0][0] - boxes[2][0]) < 1e-12 and abs(boxes[1][0] - boxes[3][0]) < 1e-12
        fig._hem_meta['paired_panel_axes'] = [list(b) for b in boxes]
    boundary = fig.bbox
    outside = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        bbox = text.get_window_extent(renderer)
        if bbox.x0 < boundary.x0 - 2 or bbox.y0 < boundary.y0 - 2 or bbox.x1 > boundary.x1 + 2 or (bbox.y1 > boundary.y1 + 2):
            if text.axes is not None and text.get_clip_on():
                continue
            outside.append(text.get_text())
    fig._hem_meta['canvas_overflow_text'] = outside
    visible_text = []
    for text in fig.findobj(mpl.text.Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        if text.axes is not None and text.get_clip_on():
            continue
        off_axis_tick = False
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                for tick in coordinate.get_major_ticks():
                    if text in (tick.label1, tick.label2) and (not min(limits) <= tick.get_loc() <= max(limits)):
                        off_axis_tick = True
        if not off_axis_tick:
            visible_text.append(text)
    collisions = []
    boxes = [text.get_window_extent(renderer) for text in visible_text]
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            if min(first.x1, second.x1) - max(first.x0, second.x0) > 0.5 and min(first.y1, second.y1) - max(first.y0, second.y0) > 0.5:
                collisions.append([visible_text[i].get_text(), visible_text[j].get_text()])
    fig._hem_meta['text_collisions'] = collisions
    fig._hem_meta['typography'] = {'family': 'Times New Roman', 'normal_pt': 9, 'panel_title_pt': 10, 'panel_number_format': '(a)'}
    fig._hem_meta['figure_size_in'] = list(fig.get_size_inches())
    fig._hem_meta['input_sha256'] = dict(s.used)
    fig._hem_meta['notes'] = 'Input checksums and plotting definitions accompany each figure.'
    for fmt in formats:
        opts = {'dpi': dpi}
        if fmt == 'pdf':
            opts['metadata'] = {'Title': f'HEM Figure {number}: data-faithful style recreation', 'Creator': 'HEM Matplotlib renderer'}
        fig.savefig(base.with_suffix('.' + fmt), **opts)
    base.with_suffix('.json').write_text(json.dumps(main__jsonable(fig._hem_meta), indent=2, ensure_ascii=False), encoding='utf-8')
    main_LOG.info('Figure %s saved: %s | overflow=%s', number, ', '.join(formats), outside)
    return base.with_suffix('.png')

def main_draw(number: int, data_root: Path | None=None, output: Path | None=None, strict_raw=True, formats=None, dpi=None):
    if number not in main_FIGURES:
        raise ValueError('Figure number must be 1–6')
    s = main_Study(data_root or ROOT / 'Data_for_Figures', output or ROOT / 'output/publication/main', strict_raw)
    f = main_FIGURES[number](s)
    path = main_export(f, number, s, formats or ['png', 'svg', 'pdf'], dpi or STYLE['dpi'])
    plt.close(f)
    (s.output / 'checks').mkdir(exist_ok=True)
    (s.output / 'checks' / f'Figure_{number:02}.json').write_text(json.dumps(main__jsonable(s.checks), indent=2), encoding='utf-8')
    return path

# Read the data, draw this figure and save it.
path = main_draw(6, data_root=DATA_ROOT, output=OUTPUT / "main", dpi=DPI)
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Extended Data Figure 1


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from importlib.resources import files
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.text import Text
import pandas as pd
from matplotlib.ticker import MaxNLocator
import string
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_ecdf(ax, values, color, label=None):
    values = np.sort(pd.Series(values).dropna().to_numpy())
    ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', color=color, lw=1.2, label=label)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cumulative fraction')

def si_s1(s):
    ash = s.read('demand/ASHRAE_RECORDS.csv', True)
    ct = s.read('demand/CTCD_RECORDS.csv', True)
    people = s.read('results/data_support/IDENTIFIED_AS_HRAE_RECORDS.csv')
    variance = s.read('results/observations/ASHRAE_NESTED_VARIANCE.csv', True)
    fig, (a, b, c, d) = si_canvas()
    for frame, color in [(ash, shared_BLUE), (ct, shared_PURPLE)]:
        a.hist(frame.reference_g050_c, bins=np.arange(10, 40.1, 0.5), density=True, histtype='step', color=color, linewidth=1.1)
    a.set(xlabel='Demand center (°C)', ylabel='Density (°C⁻¹)', xlim=(10, 45))
    shared_title(a, 'a', 'Record-level demand')
    coverage = people if {'records', 'lo', 'hi'}.issubset(people.columns) else people.groupby('person').agg(records=('temperature', 'size'), lo=('temperature', 'min'), hi=('temperature', 'max'))
    counts = coverage.records
    si_ecdf(b, counts, shared_GREEN)
    b.set_xlabel('Records per analysis unit')
    b.set_xscale('log')
    shared_title(b, 'b', 'Repeated-record support')
    c.scatter(coverage.records, coverage.hi - coverage.lo, s=5, color=shared_BLUE, alpha=0.35, linewidths=0)
    c.set(xlabel='Records per analysis unit', ylabel='Observed temperature\nspan (°C)', xscale='log')
    shared_title(c, 'c', 'Temperature coverage')
    x = variance[variance.component.eq('unit_group')]
    names = {'record_weight': 'Record', 'unit_weight': 'Unit', 'building_weight': 'Building', 'source_weight': 'Source'}
    for (weighting, group), paint in zip(x.groupby('weighting'), [shared_BLUE, shared_GOLD, shared_GREEN, shared_PURPLE]):
        d.plot(group.griffiths_coefficient, group.share * 100, marker='o', ms=3, lw=0.9, color=paint, label=names.get(weighting, weighting))
    d.set(xlabel='Griffiths coefficient (°C⁻¹)', ylabel='Within-building\nvariance share (%)')
    shared_title(d, 'd', 'Reconstruction sensitivity')
    d.legend(loc='upper center', bbox_to_anchor=(0.5, 1.01), ncol=2, columnspacing=0.5, handlelength=0.8)
    d.set_ylim(0, 100)
    a.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['ASHRAE', 'CTCD'], loc='upper right')
    return (fig, 'abcd')

def si_s3(s):
    fig, axes = si_canvas(6)
    files = ['results/demand_validation/FINAL_TEST_METRICS.csv', 'results/demand_validation/undated_acceptability/FINAL_METRICS.csv', 'results/demand_validation/single_record/timed/FINAL_METRICS.csv', 'results/demand_validation/single_record/undated/FINAL_METRICS.csv']
    headings = ['Later requests\nRepeated calibration', 'Acceptance\nRepeated calibration', 'Later requests\nSingle-record calibration', 'Acceptance\nSingle-record calibration']
    for i, (path, heading) in enumerate(zip(files, headings)):
        x = s.read(path)
        x = x[x.status.eq('completed')].copy()
        labels = [('Center' if r.center_model == 'raw_center' else 'Shrink') + ' + ' + ('fixed' if r.range_model == 'fixed_width' else 'range') for r in x.itertuples()]
        ax = axes[i]
        ax.hlines(np.arange(len(x)), x.ba_lower95, x.ba_upper95, color='#777777', lw=0.8)
        ax.scatter(x.balanced_accuracy, np.arange(len(x)), c=[shared_BLUE if k == 'fixed_width' else shared_PURPLE for k in x.range_model], s=19, zorder=3)
        ax.set_yticks(range(len(x)), labels)
        ax.set_ylim(len(x) - 0.5, -0.5)
        ax.set(xlabel='Balanced accuracy', xlim=(0.25, 0.85))
        ax.set_xticks([0.3, 0.5, 0.7])
        ax.axvline(0.5, color='#999999', lw=0.5, ls=':')
        shared_title(ax, 'abcdef'[i], heading)
    contrasts = s.read('results/group_information_validation/ALL_PAIRED_INFORMATION_CONTRASTS.csv')
    x = contrasts[contrasts.width_policy.eq('shared_width') & contrasts.individual_model.eq('raw_center') & contrasts.group_model.eq('building_median')].sort_values('design')
    names = {'timed_multi': 'Later, repeated', 'timed_single': 'Later, single', 'undated_multi': 'Acceptance, repeated', 'undated_single': 'Acceptance, single'}
    axes[4].hlines(range(len(x)), x.paired_ba_difference_lower95, x.paired_ba_difference_upper95, color=shared_BLUE, lw=1)
    axes[4].scatter(x.balanced_accuracy_difference, range(len(x)), s=18, color=shared_PURPLE)
    axes[4].set_yticks(range(len(x)), [names[z].replace(', ', '\n') for z in x.design])
    axes[4].invert_yaxis()
    axes[4].axvline(0, color='#777777', lw=0.6)
    axes[4].set_xlabel('Accuracy difference')
    shared_title(axes[4], 'e', 'Individual minus building')
    proxy = s.read('results/demand_validation/single_record/timed/CONTINUOUS_PROXY_METRICS.csv')
    y = np.arange(len(proxy))
    axes[5].hlines(y, proxy.mae_lower95_c, proxy.mae_upper95_c, color=shared_BLUE, lw=1)
    axes[5].scatter(proxy.mean_absolute_unit_proxy_error_c, y, s=20, color=shared_PURPLE)
    axes[5].set_yticks(y, ['Individual' if n == 'raw_center' else 'Shrinkage' for n in proxy.center_model])
    axes[5].invert_yaxis()
    axes[5].set_xlabel('Later-center proxy MAE (°C)')
    shared_title(axes[5], 'f', 'Center prediction error')
    return (fig, 'abcdef')

si_FUNCTIONS = {1: si_s1, 3: si_s3}

extended_GROUPS = {1: (1, 3)}

def extended_draw(number, output):
    global si_canvas
    fonts = shared_style()
    fig = plt.figure(figsize=(179 / 25.4, 210 / 25.4))
    source = si_Sources(ROOT / 'Data_for_Figures/output/from_raw', ROOT / 'Data_for_Figures/output/strengthening')
    panels, legends, colorbars, origins = ([], [], [], [])
    original_canvas = si_canvas

    def canvas(count=4, legend_gap=False):
        axes = [fig.add_axes([0.1, 0.1, 0.3, 0.2]) for _ in range(count)]
        panels.extend(axes)
        return (fig, axes)
    try:
        si_canvas = canvas
        for old_number in extended_GROUPS[number]:
            before_axes, before_legends, before_panels = (len(fig.axes), len(fig.legends), len(panels))
            si_FUNCTIONS[old_number](source)
            new_panels = panels[before_panels:]
            origins.extend(((old_number, letter) for letter in string.ascii_lowercase[:len(new_panels)]))
            legends.extend(((old_number, legend) for legend in fig.legends[before_legends:]))
            colorbars.extend(((old_number, ax) for ax in fig.axes[before_axes:] if ax not in new_panels))
    finally:
        si_canvas = original_canvas
    if number in (1, 3):
        order = [8, 9, 4, 5, 6, 7, 0, 1, 2, 3] if number == 1 else list(range(8))
        top = [[0.145, 0.765, 0.34, 0.18], [0.645, 0.765, 0.34, 0.18]]
        slots = top + [[x, y, 0.19, 0.145] for y in [0.525, 0.275] for x in [0.145, 0.465, 0.785]] + [[0.145, 0.055, 0.34, 0.145], [0.645, 0.055, 0.34, 0.145]]
        if number == 3:
            top = [[0.13, 0.765, 0.32, 0.18], [0.63, 0.765, 0.32, 0.18]]
            slots = top + [[x, 0.445, 0.19, 0.2] for x in [0.145, 0.465, 0.785]] + [[x, 0.115, 0.19, 0.23] for x in [0.145, 0.465, 0.785]]
    else:
        order = list(range(len(panels)))
        slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.565, 0.33, 0.095] for x in [0.145, 0.465, 0.785]]
        if number == 4:
            slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.55, 0.32, 0.09] for x in [0.145, 0.465, 0.785]]
    panel_map = {}
    for letter, index, bounds in zip(string.ascii_lowercase, order, slots):
        ax = panels[index]
        ax.set_position(bounds)
        heading = ax.get_title(loc='left').split(' ', 1)[1]
        heading = heading.replace('\n', ': ')
        short = {'Later requests: Repeated calibration': 'Later requests, repeated', 'Acceptance: Repeated calibration': 'Acceptance, repeated', 'Later requests: Single-record calibration': 'Later requests, single', 'Acceptance: Single-record calibration': 'Acceptance, single', 'Individual minus building': 'Individual information', 'Center prediction error': 'Later-center error', 'Magnitude of exceedance': 'Service exceedance', 'Feasible settings out of 54': 'Service frontier', 'Annual operation support': 'Operation support', 'End-use relationship': 'Heating and cooling', 'Mean choices exceeding individual service limits': 'Service exceedances', 'Where averaging matters': 'Service exceedances', 'Hourly electricity residual': 'Electricity residual', 'Repeated-record support': 'Repeated records'}.get(heading, heading)
        renderer = fig.canvas.get_renderer()
        heading_width = renderer.get_text_width_height_descent(f'({letter}) {short}', FontProperties(family='Times New Roman', size=10, weight='bold'), False)[0]
        row_neighbors = [slot[0] for slot in slots if abs(slot[1] - bounds[1]) < 1e-10 and slot[0] > bounds[0]]
        heading_right = min(row_neighbors) - 0.012 if row_neighbors else 0.993
        available = (heading_right - bounds[0]) * fig.bbox.width
        if heading_width > available:
            words = short.split()
            middle = min(range(1, len(words)), key=lambda i: abs(len(' '.join(words[:i])) - len(' '.join(words[i:]))))
            short = ' '.join(words[:middle]) + '\n' + ' '.join(words[middle:])
        shared_title(ax, letter, short)
        panel_map[letter] = {'source_figure': origins[index][0], 'source_panel': origins[index][1]}
        if ax.get_xlabel() == 'Choice-disagreement frequency':
            ax.set_xlabel('Choice-disagreement\nfrequency')
        if ax.get_xlabel() == 'Reassignment allowance (%)':
            ax.set_xlabel('Reassignment\nallowance (%)')
        ax.tick_params(axis='both', pad=2)
        if ax.get_xscale() == 'linear' and len(ax.get_xticks()) > 6:
            ax.xaxis.set_major_locator(MaxNLocator(3))
        if origins[index][0] == 3 and origins[index][1] in 'abcd':
            ax.set_yticks(ax.get_yticks(), [t.get_text().replace(' + ', '\n+ ') for t in ax.get_yticklabels()])
        for coordinate, limits in [(ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())]:
            ticks = coordinate.get_majorticklocs()
            inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
            if not inside.all():
                coordinate.set_ticks(ticks[inside])
    for old_number, axis in colorbars:
        parent = panels[origins.index((old_number, 'c' if old_number == 5 else 'a'))]
        bounds = parent.get_position()
        if old_number == 5:
            parent.set_position([bounds.x0, bounds.y0, bounds.width - 0.035, bounds.height])
            bounds = parent.get_position()
        axis.set_position([bounds.x1 + 0.012, bounds.y0, 0.012, bounds.height])
    for old_number, legend in legends:
        if number == 2 and old_number == 6:
            panels[6].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper left', handlelength=0.8)
            legend.remove()
            continue
        legend.set_loc('center')
        legend.set_ncols(2 if len(legend.get_texts()) == 2 else 3)
        legend.set_bbox_to_anchor({1: (0.55, 0.018), 2: (0.55, 0.502 if old_number == 6 else 0.027), 3: (0.55, 0.695 if old_number == 8 else 0.035), 4: (0.55, 0.73)}[number], transform=fig.transFigure)
    if number == 1:
        panels[0].set_xlim(10, 55)
        legend = panels[0].get_legend()
        panels[0].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper right', handlelength=0.75, handletextpad=0.35, borderaxespad=0.25)
    if number == 4:
        panels[5].set_position([0.785, 0.535, 0.19, 0.145])
        panel = panels[6]
        panel.set_ylim(0, 1.6)
        panel.get_legend().set_loc('upper left')
        panel.tick_params(axis='x', pad=5)
        for line in panel.lines:
            if len(line.get_xdata()) == 2 and all((x == 0 for x in line.get_xdata())):
                line.set_ydata([0, 1 / 1.6])
        colorbars[0][1].set_yticks([0, 200])
    if number == 3:
        panels[7].set_xlim(0.94, 1.06)
        panels[7].set_ylim(0, 1.5)
        panels[7].get_legend().set_loc('upper right')
    if number == 2:
        panels[0].set_ylim(0, 100)
        lo, hi = panels[6].get_ylim()
        panels[6].set_ylim(lo, hi + 0.5 * (hi - lo))
    if number in (2, 4):
        matrix_axis = panels[2 if number == 2 else 4]
        values = matrix_axis.images[0].get_array()
        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                value = int(values[row, column])
                if value:
                    matrix_axis.text(column, row, str(value), ha='center', va='center', color='white' if value > values.max() * 0.6 else 'black')
    output = Path(output)
    if not output.resolve().is_relative_to(ROOT):
        raise ValueError('Publication output must remain inside the project')
    try:
        shared_export(fig, output, 200 + number, source.paths, string.ascii_lowercase[:len(panels)], fonts)
    finally:
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = output / f'Figure_{200 + number}.{extension}'
            if old.exists():
                old.replace(output / f'Figure_E{number:02}.{extension}')
    receipt_path = output / f'Figure_E{number:02}.json'
    receipt = json.loads(receipt_path.read_text())
    receipt.update({'panel_sources': panel_map, 'extended_plot_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), 'figure_width_in': 179 / 25.4, 'figure_height_in': 210 / 25.4})
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
    return output / f'Figure_E{number:02}.png'

# Read the data, draw this figure and save it.
path = extended_draw(1, OUTPUT / "extended")
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Extended Data Figure 2


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.ticker import MaxNLocator
import string
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_ecdf(ax, values, color, label=None):
    values = np.sort(pd.Series(values).dropna().to_numpy())
    ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', color=color, lw=1.2, label=label)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cumulative fraction')

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s5(s):
    levels = s.read('results/aligned_grid_sensitivity/ALIGNED_GRID_SUMMARY.csv')
    changes = s.read('results/aligned_grid_sensitivity/ALIGNED_GRID_LEVEL_CHANGES.csv')
    budgets = s.read('results/budget_frontiers/COMMON_BUDGET_MINIMUM_ENERGY.csv')
    tol = s.read('results/decision_tolerance_sensitivity/DECISION_TOLERANCE_SUMMARY.csv', True)
    fig, (a, b, c, d) = si_canvas()
    for col, label, paint in [('qualified_set_changes', 'Qualifying set', shared_PURPLE), ('choice_changes', 'Energy choice', shared_BLUE)]:
        a.plot(range(len(levels)), levels[col], marker='o', color=paint, lw=1, label=label)
    a.set_xticks(range(len(levels)), ['0.5', '0.25', '0.125'])
    a.set(xlabel='Aligned offset grid (°C)', ylabel='Settings with a change', ylim=(0, 85))
    a.legend(loc='upper center', ncol=1, handlelength=0.8)
    shared_title(a, 'a', 'Choices across grids')
    for level, g in changes.groupby('to_level'):
        si_ecdf(b, -g.full_selected_energy_change_pct, shared_BLUE if len(b.lines) == 0 else shared_PURPLE, {'eighth': '0.125 °C', 'quarter': '0.25 °C'}[str(level)])
    b.set_xlabel('Energy improvement (%)')
    shared_title(b, 'b', 'Refining energy minima')
    b.legend(loc='lower right', handlelength=0.8)
    x = budgets[budgets.budget_type.eq('absolute_mismatch') & budgets.permission.eq('native_zone')]
    if 'full' in x.representation.unique():
        x = x[x.representation.eq('full')]
    table = x.assign(feasible=x.status.eq('feasible')).groupby(['fixed_tail_budget_c', 'population_budget_c']).feasible.sum().unstack()
    if table.empty:
        raise ValueError('Absolute service grid missing')
    image = c.imshow(table.to_numpy(), origin='lower', aspect='auto', cmap='Purples', vmin=0, vmax=54)
    c.set_xticks(range(len(table.columns)), [f'{z:g}' for z in table.columns])
    c.set_yticks(range(len(table)), [f'{z:g}' for z in table.index])
    c.set(xlabel='Population budget (°C)', ylabel='Least-served budget (°C)')
    shared_title(c, 'c', 'Feasible settings out of 54')
    fig.colorbar(image, cax=fig.add_axes([0.49, 0.15, 0.014, 0.29]), ticks=[0, 27, 54])
    d.plot(tol.delta_c, tol.eligibility_set_changed_settings, marker='o', ms=3, color=shared_PURPLE)
    d.plot(tol.delta_c, tol.selected_physical_plan_changed_settings, marker='o', ms=3, color=shared_BLUE)
    d.set(xlabel='Added service margin (°C)', ylabel='Settings with a change', ylim=(0, 30))
    shared_title(d, 'd', 'Original-library margins')
    return (fig, 'abcd')

def si_s6(s):
    x = s.read('results/information_uncertainty_summary/SETTING_SELECTION_STABILITY_ALL_BRANCHES.csv')
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    branch = x.library_and_scenario_design.unique()
    for label, paint in [('pooled_archive_shared', shared_BLUE), ('pooled_study_shared', shared_PURPLE)]:
        names = [z for z in branch if ('study' in z) == ('study' in label) and 'pooled' in z and ('temporal' not in z) and ('single' not in z)]
        if not names:
            continue
        g = x[x.library_and_scenario_design.eq(names[0]) & x.scenario.str.contains('SD0.5_personSD0.5', regex=False)]
        if g.empty:
            continue
        si_ecdf(a, g.conditional_choice_disagreement_fraction, paint)
        si_ecdf(b, g.zone_mean_selected_excess_p95_c, paint)
        c.scatter(g.conditional_choice_disagreement_fraction, g.extra_selected_energy_median_kwh / 1000, color=paint, s=12, alpha=0.6)
    for name in [z for z in branch if 'temporal' in z or 'single' in z]:
        g = x[x.library_and_scenario_design.eq(name) & x.scenario.eq('observed_temporal_proxy_shift_fraction1')]
        si_ecdf(d, g.conditional_choice_disagreement_fraction, shared_BLUE if 'single' in name else shared_PURPLE)
    a.set_xlabel('Choice-disagreement frequency')
    shared_title(a, 'a', 'Shared-error structures')
    b.set_xlabel('95th-percentile service\nexceedance (°C)')
    shared_title(b, 'b', 'Magnitude of exceedance')
    c.set(xlabel='Choice-disagreement frequency', ylabel='Median extra energy (MWh)')
    shared_title(c, 'c', 'Resource consequence')
    d.set_xlabel('Choice-disagreement\nfrequency')
    shared_title(d, 'd', 'Temporal error scenarios')
    if not a.lines or not d.lines:
        raise ValueError('Expected uncertainty branches not found')
    si_common_legend(fig, ['Archive shared', 'Contributor shared'], [shared_BLUE, shared_PURPLE], y=0.5)
    d.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['Single record', 'Repeated records'], loc='lower right', handlelength=0.8)
    return (fig, 'abcd')

def si_s7(s):
    x = s.read('results/constrained_placement/PAIRED_WITHIN_FLOOR_PLACEMENT_EFFECTS.csv')
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    budgets = sorted(x.maximum_relocation_fraction.unique())
    for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
        g = x[x.control.eq(control)]
        for ax, column in [(a, 'actual_relocation_fraction'), (b, 'population_gain_c'), (c, 'fixed_tail_gain_c'), (d, 'energy_change_pct')]:
            for _, setting in g.groupby(['climate', 'composition_id']):
                setting = setting.sort_values('maximum_relocation_fraction')
                multiplier = 100 if column == 'actual_relocation_fraction' else 1
                ax.plot(setting.maximum_relocation_fraction * 100, setting[column] * multiplier, color=paint, lw=0.45, alpha=0.16)
            median = g.groupby('maximum_relocation_fraction')[column].median()
            ax.plot(median.index * 100, median * (100 if column == 'actual_relocation_fraction' else 1), color=paint, lw=1.5, marker='o', ms=3)
    for ax in [a, b, c, d]:
        ax.set_xticks([v * 100 for v in budgets])
        ax.set_xlabel('Reassignment allowance (%)')
    a.set_ylabel('People reassigned (%)')
    b.set_ylabel('Population mismatch\nreduction (°C)')
    c.set_ylabel('Least-served mismatch\nreduction (°C)')
    d.set_ylabel('Purchased-energy change (%)')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Actual movement', 'Population service', 'Least-served service', 'Energy consequence']):
        shared_title(ax, letter, name)
    si_common_legend(fig, ['Personal center', 'Personal range'], [shared_BLUE, shared_PURPLE], y=0.5)
    return (fig, 'abcd')

si_FUNCTIONS = {5: si_s5, 6: si_s6, 7: si_s7}

extended_GROUPS = {2: (5, 6, 7)}

def extended_draw(number, output):
    global si_canvas
    fonts = shared_style()
    fig = plt.figure(figsize=(179 / 25.4, 210 / 25.4))
    source = si_Sources(ROOT / 'Data_for_Figures/output/from_raw', ROOT / 'Data_for_Figures/output/strengthening')
    panels, legends, colorbars, origins = ([], [], [], [])
    original_canvas = si_canvas

    def canvas(count=4, legend_gap=False):
        axes = [fig.add_axes([0.1, 0.1, 0.3, 0.2]) for _ in range(count)]
        panels.extend(axes)
        return (fig, axes)
    try:
        si_canvas = canvas
        for old_number in extended_GROUPS[number]:
            before_axes, before_legends, before_panels = (len(fig.axes), len(fig.legends), len(panels))
            si_FUNCTIONS[old_number](source)
            new_panels = panels[before_panels:]
            origins.extend(((old_number, letter) for letter in string.ascii_lowercase[:len(new_panels)]))
            legends.extend(((old_number, legend) for legend in fig.legends[before_legends:]))
            colorbars.extend(((old_number, ax) for ax in fig.axes[before_axes:] if ax not in new_panels))
    finally:
        si_canvas = original_canvas
    if number in (1, 3):
        order = [8, 9, 4, 5, 6, 7, 0, 1, 2, 3] if number == 1 else list(range(8))
        top = [[0.145, 0.765, 0.34, 0.18], [0.645, 0.765, 0.34, 0.18]]
        slots = top + [[x, y, 0.19, 0.145] for y in [0.525, 0.275] for x in [0.145, 0.465, 0.785]] + [[0.145, 0.055, 0.34, 0.145], [0.645, 0.055, 0.34, 0.145]]
        if number == 3:
            top = [[0.13, 0.765, 0.32, 0.18], [0.63, 0.765, 0.32, 0.18]]
            slots = top + [[x, 0.445, 0.19, 0.2] for x in [0.145, 0.465, 0.785]] + [[x, 0.115, 0.19, 0.23] for x in [0.145, 0.465, 0.785]]
    else:
        order = list(range(len(panels)))
        slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.565, 0.33, 0.095] for x in [0.145, 0.465, 0.785]]
        if number == 4:
            slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.55, 0.32, 0.09] for x in [0.145, 0.465, 0.785]]
    panel_map = {}
    for letter, index, bounds in zip(string.ascii_lowercase, order, slots):
        ax = panels[index]
        ax.set_position(bounds)
        heading = ax.get_title(loc='left').split(' ', 1)[1]
        heading = heading.replace('\n', ': ')
        short = {'Later requests: Repeated calibration': 'Later requests, repeated', 'Acceptance: Repeated calibration': 'Acceptance, repeated', 'Later requests: Single-record calibration': 'Later requests, single', 'Acceptance: Single-record calibration': 'Acceptance, single', 'Individual minus building': 'Individual information', 'Center prediction error': 'Later-center error', 'Magnitude of exceedance': 'Service exceedance', 'Feasible settings out of 54': 'Service frontier', 'Annual operation support': 'Operation support', 'End-use relationship': 'Heating and cooling', 'Mean choices exceeding individual service limits': 'Service exceedances', 'Where averaging matters': 'Service exceedances', 'Hourly electricity residual': 'Electricity residual', 'Repeated-record support': 'Repeated records'}.get(heading, heading)
        renderer = fig.canvas.get_renderer()
        heading_width = renderer.get_text_width_height_descent(f'({letter}) {short}', FontProperties(family='Times New Roman', size=10, weight='bold'), False)[0]
        row_neighbors = [slot[0] for slot in slots if abs(slot[1] - bounds[1]) < 1e-10 and slot[0] > bounds[0]]
        heading_right = min(row_neighbors) - 0.012 if row_neighbors else 0.993
        available = (heading_right - bounds[0]) * fig.bbox.width
        if heading_width > available:
            words = short.split()
            middle = min(range(1, len(words)), key=lambda i: abs(len(' '.join(words[:i])) - len(' '.join(words[i:]))))
            short = ' '.join(words[:middle]) + '\n' + ' '.join(words[middle:])
        shared_title(ax, letter, short)
        panel_map[letter] = {'source_figure': origins[index][0], 'source_panel': origins[index][1]}
        if ax.get_xlabel() == 'Choice-disagreement frequency':
            ax.set_xlabel('Choice-disagreement\nfrequency')
        if ax.get_xlabel() == 'Reassignment allowance (%)':
            ax.set_xlabel('Reassignment\nallowance (%)')
        ax.tick_params(axis='both', pad=2)
        if ax.get_xscale() == 'linear' and len(ax.get_xticks()) > 6:
            ax.xaxis.set_major_locator(MaxNLocator(3))
        if origins[index][0] == 3 and origins[index][1] in 'abcd':
            ax.set_yticks(ax.get_yticks(), [t.get_text().replace(' + ', '\n+ ') for t in ax.get_yticklabels()])
        for coordinate, limits in [(ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())]:
            ticks = coordinate.get_majorticklocs()
            inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
            if not inside.all():
                coordinate.set_ticks(ticks[inside])
    for old_number, axis in colorbars:
        parent = panels[origins.index((old_number, 'c' if old_number == 5 else 'a'))]
        bounds = parent.get_position()
        if old_number == 5:
            parent.set_position([bounds.x0, bounds.y0, bounds.width - 0.035, bounds.height])
            bounds = parent.get_position()
        axis.set_position([bounds.x1 + 0.012, bounds.y0, 0.012, bounds.height])
    for old_number, legend in legends:
        if number == 2 and old_number == 6:
            panels[6].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper left', handlelength=0.8)
            legend.remove()
            continue
        legend.set_loc('center')
        legend.set_ncols(2 if len(legend.get_texts()) == 2 else 3)
        legend.set_bbox_to_anchor({1: (0.55, 0.018), 2: (0.55, 0.502 if old_number == 6 else 0.027), 3: (0.55, 0.695 if old_number == 8 else 0.035), 4: (0.55, 0.73)}[number], transform=fig.transFigure)
    if number == 1:
        panels[0].set_xlim(10, 55)
        legend = panels[0].get_legend()
        panels[0].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper right', handlelength=0.75, handletextpad=0.35, borderaxespad=0.25)
    if number == 4:
        panels[5].set_position([0.785, 0.535, 0.19, 0.145])
        panel = panels[6]
        panel.set_ylim(0, 1.6)
        panel.get_legend().set_loc('upper left')
        panel.tick_params(axis='x', pad=5)
        for line in panel.lines:
            if len(line.get_xdata()) == 2 and all((x == 0 for x in line.get_xdata())):
                line.set_ydata([0, 1 / 1.6])
        colorbars[0][1].set_yticks([0, 200])
    if number == 3:
        panels[7].set_xlim(0.94, 1.06)
        panels[7].set_ylim(0, 1.5)
        panels[7].get_legend().set_loc('upper right')
    if number == 2:
        panels[0].set_ylim(0, 100)
        lo, hi = panels[6].get_ylim()
        panels[6].set_ylim(lo, hi + 0.5 * (hi - lo))
    if number in (2, 4):
        matrix_axis = panels[2 if number == 2 else 4]
        values = matrix_axis.images[0].get_array()
        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                value = int(values[row, column])
                if value:
                    matrix_axis.text(column, row, str(value), ha='center', va='center', color='white' if value > values.max() * 0.6 else 'black')
    output = Path(output)
    if not output.resolve().is_relative_to(ROOT):
        raise ValueError('Publication output must remain inside the project')
    try:
        shared_export(fig, output, 200 + number, source.paths, string.ascii_lowercase[:len(panels)], fonts)
    finally:
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = output / f'Figure_{200 + number}.{extension}'
            if old.exists():
                old.replace(output / f'Figure_E{number:02}.{extension}')
    receipt_path = output / f'Figure_E{number:02}.json'
    receipt = json.loads(receipt_path.read_text())
    receipt.update({'panel_sources': panel_map, 'extended_plot_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), 'figure_width_in': 179 / 25.4, 'figure_height_in': 210 / 25.4})
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
    return output / f'Figure_E{number:02}.png'

# Read the data, draw this figure and save it.
path = extended_draw(2, OUTPUT / "extended")
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Extended Data Figure 3


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.ticker import MaxNLocator
import string
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_ecdf(ax, values, color, label=None):
    values = np.sort(pd.Series(values).dropna().to_numpy())
    ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', color=color, lw=1.2, label=label)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cumulative fraction')

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s8(s):
    x = s.read('results/hvac_retuned/WITHIN_SYSTEM_STRATEGY_COMPARISONS.csv')
    operations = s.read('results/hvac_retuned/SYSTEM_OPERATION_SCORES.csv')
    unavailable = s.read('results/hvac_retuned/UNAVAILABLE_RETUNED_SETTINGS.csv')
    x = x[x.engineering_pass_demand.eq(True) & x.engineering_pass_average.eq(True)]
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
        g = x[x.compared_control.eq(control)]
        a.scatter(g.population_mismatch_reduction_c, g.delta_purchased_energy_pct, s=15, color=paint, alpha=0.65)
        b.scatter(g.fixed_tail_mismatch_reduction_c, g.delta_purchased_energy_pct, s=15, color=paint, alpha=0.65)
        si_ecdf(c, g.heating_unmet_zone_hours_demand + g.cooling_unmet_zone_hours_demand, paint)
    a.set(xlabel='Population mismatch\nreduction (°C)', ylabel='Purchased-energy change (%)')
    b.set(xlabel='Least-served mismatch\nreduction (°C)', ylabel='Purchased-energy change (%)')
    c.set_xlabel('Unmet heating and cooling\n(zone-hours)')
    counts = operations.groupby('climate').engineering_pass.agg(['sum', 'size']).sort_index()
    for climate, g in unavailable.groupby('climate'):
        if climate not in counts.index:
            counts.loc[climate] = [0, len(g)]
        else:
            counts.loc[climate, 'size'] += len(g)
    counts = counts.sort_index()
    positions = np.arange(len(counts))
    d.barh(positions, counts['size'], color='#e8e8e8', height=0.6)
    d.barh(positions, counts['sum'], color=shared_GREEN, height=0.6)
    d.set_yticks(positions, [str(v).replace('_', ' ').title() for v in counts.index])
    d.invert_yaxis()
    d.set_xlabel('Annual operations')
    d.xaxis.set_major_locator(MaxNLocator(3, integer=True))
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Population service', 'Least-served service', 'Load tracking', 'Annual operation support']):
        shared_title(ax, letter, name)
    si_common_legend(fig, ['Personal center', 'Personal range'], [shared_BLUE, shared_PURPLE], y=0.5)
    d.set_ylim(len(counts) - 0.5, -3.2)
    d.legend([Patch(fc='#e8e8e8'), Patch(fc=shared_GREEN)], ['Planned', 'Engineering accepted'], loc='upper center', bbox_to_anchor=(0.5, 1.03), handlelength=0.8)
    return (fig, 'abcd')

def si_s9(s):
    energy = s.latest('results/spatial_hvac_mechanism', 'ENERGY_WINDOW_LEVELS.csv')
    tracking = s.latest('results/spatial_hvac_mechanism', 'TRACKING_WINDOW_LEVELS.csv')
    vrp = s.latest('results/vrp_request_diagnostic', 'AIRLOOP_REQUEST_FULFILLMENT.csv')
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    keys = ['climate', 'composition_id', 'control', 'system', 'placement']
    annual = energy.groupby(keys, as_index=False).sum(numeric_only=True)
    tracking = tracking.groupby(keys, as_index=False).sum(numeric_only=True)
    systems = sorted(annual.system.unique())
    for system, paint in zip(systems, [shared_BLUE, shared_PURPLE]):
        g = annual[annual.system.eq(system)]
        a.scatter(g.cooling_electric_kwh / 1000, g.heating_electric_kwh / 1000, s=10, color=paint, alpha=0.4)
        si_ecdf(b, g.joint_heating_cooling_hour_bins, paint)
        t = tracking[tracking.system.eq(system)]
        si_ecdf(c, t.heating_unmet_zone_hours + t.cooling_unmet_zone_hours, paint)
    a.set(xlabel='Cooling electricity (MWh)', ylabel='Heating electricity (MWh)')
    b.set_xlabel('Hours with heating\nand cooling')
    c.set_xlabel('Unmet heating and cooling\n(zone-hours)')
    valid = vrp.dropna(subset=['actual_to_vrp_request_ratio_min'])
    for placement, paint in [('mixed', shared_BLUE), ('arranged', shared_PURPLE)]:
        g = valid[valid.placement.eq(placement)]
        if len(g):
            si_ecdf(d, g.actual_to_vrp_request_ratio_min, paint)
    d.set_xlabel('Minimum outdoor-air\ndelivery / native request')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['End-use relationship', 'Coincident operation', 'Annual load tracking', 'Outdoor-air request']):
        shared_title(ax, letter, name)
    si_common_legend(fig, [{'pthp': 'Zone heat pump', 'vav': 'VAV'}[v] for v in systems], [shared_BLUE, shared_PURPLE], y=0.5)
    d.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['Mixed', 'Arranged'], loc='lower right', handlelength=0.7)
    return (fig, 'abcd')

si_FUNCTIONS = {8: si_s8, 9: si_s9}

extended_GROUPS = {3: (8, 9)}

def extended_draw(number, output):
    global si_canvas
    fonts = shared_style()
    fig = plt.figure(figsize=(179 / 25.4, 210 / 25.4))
    source = si_Sources(ROOT / 'Data_for_Figures/output/from_raw', ROOT / 'Data_for_Figures/output/strengthening')
    panels, legends, colorbars, origins = ([], [], [], [])
    original_canvas = si_canvas

    def canvas(count=4, legend_gap=False):
        axes = [fig.add_axes([0.1, 0.1, 0.3, 0.2]) for _ in range(count)]
        panels.extend(axes)
        return (fig, axes)
    try:
        si_canvas = canvas
        for old_number in extended_GROUPS[number]:
            before_axes, before_legends, before_panels = (len(fig.axes), len(fig.legends), len(panels))
            si_FUNCTIONS[old_number](source)
            new_panels = panels[before_panels:]
            origins.extend(((old_number, letter) for letter in string.ascii_lowercase[:len(new_panels)]))
            legends.extend(((old_number, legend) for legend in fig.legends[before_legends:]))
            colorbars.extend(((old_number, ax) for ax in fig.axes[before_axes:] if ax not in new_panels))
    finally:
        si_canvas = original_canvas
    if number in (1, 3):
        order = [8, 9, 4, 5, 6, 7, 0, 1, 2, 3] if number == 1 else list(range(8))
        top = [[0.145, 0.765, 0.34, 0.18], [0.645, 0.765, 0.34, 0.18]]
        slots = top + [[x, y, 0.19, 0.145] for y in [0.525, 0.275] for x in [0.145, 0.465, 0.785]] + [[0.145, 0.055, 0.34, 0.145], [0.645, 0.055, 0.34, 0.145]]
        if number == 3:
            top = [[0.13, 0.765, 0.32, 0.18], [0.63, 0.765, 0.32, 0.18]]
            slots = top + [[x, 0.445, 0.19, 0.2] for x in [0.145, 0.465, 0.785]] + [[x, 0.115, 0.19, 0.23] for x in [0.145, 0.465, 0.785]]
    else:
        order = list(range(len(panels)))
        slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.565, 0.33, 0.095] for x in [0.145, 0.465, 0.785]]
        if number == 4:
            slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.55, 0.32, 0.09] for x in [0.145, 0.465, 0.785]]
    panel_map = {}
    for letter, index, bounds in zip(string.ascii_lowercase, order, slots):
        ax = panels[index]
        ax.set_position(bounds)
        heading = ax.get_title(loc='left').split(' ', 1)[1]
        heading = heading.replace('\n', ': ')
        short = {'Later requests: Repeated calibration': 'Later requests, repeated', 'Acceptance: Repeated calibration': 'Acceptance, repeated', 'Later requests: Single-record calibration': 'Later requests, single', 'Acceptance: Single-record calibration': 'Acceptance, single', 'Individual minus building': 'Individual information', 'Center prediction error': 'Later-center error', 'Magnitude of exceedance': 'Service exceedance', 'Feasible settings out of 54': 'Service frontier', 'Annual operation support': 'Operation support', 'End-use relationship': 'Heating and cooling', 'Mean choices exceeding individual service limits': 'Service exceedances', 'Where averaging matters': 'Service exceedances', 'Hourly electricity residual': 'Electricity residual', 'Repeated-record support': 'Repeated records'}.get(heading, heading)
        renderer = fig.canvas.get_renderer()
        heading_width = renderer.get_text_width_height_descent(f'({letter}) {short}', FontProperties(family='Times New Roman', size=10, weight='bold'), False)[0]
        row_neighbors = [slot[0] for slot in slots if abs(slot[1] - bounds[1]) < 1e-10 and slot[0] > bounds[0]]
        heading_right = min(row_neighbors) - 0.012 if row_neighbors else 0.993
        available = (heading_right - bounds[0]) * fig.bbox.width
        if heading_width > available:
            words = short.split()
            middle = min(range(1, len(words)), key=lambda i: abs(len(' '.join(words[:i])) - len(' '.join(words[i:]))))
            short = ' '.join(words[:middle]) + '\n' + ' '.join(words[middle:])
        shared_title(ax, letter, short)
        panel_map[letter] = {'source_figure': origins[index][0], 'source_panel': origins[index][1]}
        if ax.get_xlabel() == 'Choice-disagreement frequency':
            ax.set_xlabel('Choice-disagreement\nfrequency')
        if ax.get_xlabel() == 'Reassignment allowance (%)':
            ax.set_xlabel('Reassignment\nallowance (%)')
        ax.tick_params(axis='both', pad=2)
        if ax.get_xscale() == 'linear' and len(ax.get_xticks()) > 6:
            ax.xaxis.set_major_locator(MaxNLocator(3))
        if origins[index][0] == 3 and origins[index][1] in 'abcd':
            ax.set_yticks(ax.get_yticks(), [t.get_text().replace(' + ', '\n+ ') for t in ax.get_yticklabels()])
        for coordinate, limits in [(ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())]:
            ticks = coordinate.get_majorticklocs()
            inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
            if not inside.all():
                coordinate.set_ticks(ticks[inside])
    for old_number, axis in colorbars:
        parent = panels[origins.index((old_number, 'c' if old_number == 5 else 'a'))]
        bounds = parent.get_position()
        if old_number == 5:
            parent.set_position([bounds.x0, bounds.y0, bounds.width - 0.035, bounds.height])
            bounds = parent.get_position()
        axis.set_position([bounds.x1 + 0.012, bounds.y0, 0.012, bounds.height])
    for old_number, legend in legends:
        if number == 2 and old_number == 6:
            panels[6].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper left', handlelength=0.8)
            legend.remove()
            continue
        legend.set_loc('center')
        legend.set_ncols(2 if len(legend.get_texts()) == 2 else 3)
        legend.set_bbox_to_anchor({1: (0.55, 0.018), 2: (0.55, 0.502 if old_number == 6 else 0.027), 3: (0.55, 0.695 if old_number == 8 else 0.035), 4: (0.55, 0.73)}[number], transform=fig.transFigure)
    if number == 1:
        panels[0].set_xlim(10, 55)
        legend = panels[0].get_legend()
        panels[0].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper right', handlelength=0.75, handletextpad=0.35, borderaxespad=0.25)
    if number == 4:
        panels[5].set_position([0.785, 0.535, 0.19, 0.145])
        panel = panels[6]
        panel.set_ylim(0, 1.6)
        panel.get_legend().set_loc('upper left')
        panel.tick_params(axis='x', pad=5)
        for line in panel.lines:
            if len(line.get_xdata()) == 2 and all((x == 0 for x in line.get_xdata())):
                line.set_ydata([0, 1 / 1.6])
        colorbars[0][1].set_yticks([0, 200])
    if number == 3:
        panels[7].set_xlim(0.94, 1.06)
        panels[7].set_ylim(0, 1.5)
        panels[7].get_legend().set_loc('upper right')
    if number == 2:
        panels[0].set_ylim(0, 100)
        lo, hi = panels[6].get_ylim()
        panels[6].set_ylim(lo, hi + 0.5 * (hi - lo))
    if number in (2, 4):
        matrix_axis = panels[2 if number == 2 else 4]
        values = matrix_axis.images[0].get_array()
        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                value = int(values[row, column])
                if value:
                    matrix_axis.text(column, row, str(value), ha='center', va='center', color='white' if value > values.max() * 0.6 else 'black')
    output = Path(output)
    if not output.resolve().is_relative_to(ROOT):
        raise ValueError('Publication output must remain inside the project')
    try:
        shared_export(fig, output, 200 + number, source.paths, string.ascii_lowercase[:len(panels)], fonts)
    finally:
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = output / f'Figure_{200 + number}.{extension}'
            if old.exists():
                old.replace(output / f'Figure_E{number:02}.{extension}')
    receipt_path = output / f'Figure_E{number:02}.json'
    receipt = json.loads(receipt_path.read_text())
    receipt.update({'panel_sources': panel_map, 'extended_plot_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), 'figure_width_in': 179 / 25.4, 'figure_height_in': 210 / 25.4})
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
    return output / f'Figure_E{number:02}.png'

# Read the data, draw this figure and save it.
path = extended_draw(3, OUTPUT / "extended")
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Extended Data Figure 4


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.ticker import MaxNLocator
import string
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import MultipleLocator, MaxNLocator

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_ecdf(ax, values, color, label=None):
    values = np.sort(pd.Series(values).dropna().to_numpy())
    ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', color=color, lw=1.2, label=label)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cumulative fraction')

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s10(s):
    x = s.read('results/dr_capability_summary/RESOURCE_SUMMARY_BY_CLIMATE.csv')
    choices = s.read('results/dr_capability/ONE_ACTION_PER_EVENT_BUDGET_CHOICES.csv')
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    x = x[x.normal_energy_reference_factor.eq(1)]
    menus = [('building_search', 'Building', shared_GOLD), ('zone_summary', 'Zone summary', shared_BLUE), ('zone_full', 'Full zone', shared_PURPLE)]
    climates = sorted(x.climate.unique())
    for menu, label, paint in menus:
        g = x[x.control.eq(menu)].set_index('climate').reindex(climates)
        if g.available_event_power_mean_kw.isna().any():
            raise ValueError(f'Missing normal menu {menu}')
        a.plot(g.available_event_power_mean_kw, np.arange(len(climates)), marker='o', ms=3, color=paint, ls='none')
        b.plot(g.selected_positive_response_events, np.arange(len(climates)), marker='o', ms=3, color=paint, ls='none')
    for ax in [a, b]:
        ax.set_yticks(np.arange(len(climates)), [v.replace('_', ' ').title() for v in climates])
        ax.invert_yaxis()
    a.set_xlabel('All-event mean power (kW)')
    b.set_xlabel('Positive-response events')
    g = choices[choices.control.eq('building_search') & choices.normal_energy_reference_factor.eq(1) & choices.selected_action.notna()].copy()
    c.scatter(g.baseline_facility_mean_kw, g.absolute_event_facility_mean_kw, color=shared_BLUE, s=10, alpha=0.5)
    limits = [min(g.baseline_facility_mean_kw.min(), g.absolute_event_facility_mean_kw.min()), max(g.baseline_facility_mean_kw.max(), g.absolute_event_facility_mean_kw.max())]
    c.plot(limits, limits, color='#888888', ls=':', lw=0.7)
    c.set(xlabel='Normal event power (kW)', ylabel='Action event power (kW)')
    si_ecdf(d, g.event_power_reduction_kw, shared_BLUE)
    d.set_xlabel('Selected event reduction (kW)')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Available power', 'Event availability', 'Absolute event power', 'Response size']):
        shared_title(ax, letter, name)
    si_common_legend(fig, [v[1] for v in menus], [v[2] for v in menus], y=0.5)
    return (fig, 'abcd')

def si_s11(s):
    x = s.read('results/dr_information_review/PAIRED_EVENT_INFORMATION_REVIEW.csv')
    summary = s.read('results/dr_information_uncertainty/PAIRED_FROZEN_POWER_SUMMARY.csv')
    fig, (a, b, c, d) = si_canvas()
    primary = x[x.control.eq('building_search') & x.normal_energy_reference_factor.eq(1) & x.policy.eq('maximum_power')]
    if primary.empty:
        raise ValueError('Primary information comparison missing')
    primary = primary.copy()
    for representation in ['full', 'mean']:
        primary[representation + '_count'] = primary[representation + '_positive_actions'].map(lambda value: len(json.loads(value)))
    counts = primary.groupby(['full_count', 'mean_count']).size().unstack(fill_value=0).reindex(index=range(6), columns=range(6), fill_value=0)
    if counts.to_numpy().sum() != len(primary):
        raise ValueError('Action-count heatmap lost events')
    image = a.imshow(counts.to_numpy(), origin='lower', cmap='Purples', vmin=0)
    a.set(xticks=range(6), yticks=range(6), xlabel='Mean-screened actions', ylabel='Individual-screened actions')
    fig.colorbar(image, cax=fig.add_axes([0.49, 0.61, 0.014, 0.29]), ticks=[0, 100, 200])
    failed = primary.groupby('climate').mean_selected_fails_full_review.sum().sort_index()
    b.barh(np.arange(len(failed)), failed, color=shared_CORAL, height=0.65)
    b.set_yticks(np.arange(len(failed)), failed.index)
    b.invert_yaxis()
    b.set_xlabel('Mean choices exceeding\nindividual service limits')
    q = summary[summary.control.eq('building_search') & summary.budget.eq(1) & summary.support.eq('all_normal_available') & summary.stratum.eq('all')]
    for prefix, paint in [('sourceSD', shared_BLUE), ('studyGroupSD', shared_PURPLE)]:
        g = q[q.scenario.str.startswith(prefix)]
        si_ecdf(c, g.paired_mean_minus_full_power_mean_kw, paint)
    c.set_xlabel('Frozen mean minus full\navailable power (kW)')
    temporal = q[q.scenario.str.contains('temporal_proxy', regex=False)].copy().sort_values('scenario')
    names = {'observed_temporal_proxy_shift_fraction0.5': 'Repeated, half', 'observed_temporal_proxy_shift_fraction1': 'Repeated, full', 'single_record_temporal_proxy_shift_fraction0.5': 'Single, half', 'single_record_temporal_proxy_shift_fraction1': 'Single, full'}
    for i, row in enumerate(temporal.itertuples()):
        d.plot([row.paired_mean_minus_full_power_p05_kw, row.paired_mean_minus_full_power_p95_kw], [i, i], color=shared_PURPLE, lw=1.3)
        d.scatter(row.paired_mean_minus_full_power_mean_kw, i, s=16, color=shared_PURPLE)
    d.set_yticks(range(len(temporal)), [names[n] for n in temporal.scenario])
    d.invert_yaxis()
    d.set_xlabel('Frozen mean minus full\navailable power (kW)')
    for ax in [c, d]:
        ax.axvline(0, color='#888888', ls=':', lw=0.7)
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Available action counts', 'Where averaging matters', 'Shared-error structures', 'Temporal error scenarios']):
        shared_title(ax, letter, name)
    c.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['Archive shared', 'Contributor shared'], loc='lower right', handlelength=0.8)
    return (fig, 'abcd')

def si_s12(s):
    hourly = s.read('results/dr_terminal_recovery/PHYSICAL_PAIR_HOURLY_RESIDUALS.csv')
    totals = s.read('results/dr_terminal_recovery/PHYSICAL_PAIR_POST_RECOVERY_SUMMARY.csv')
    fig, (a, b, c, d) = si_canvas()
    terminal = hourly.sort_values('audit_hour_index').groupby(['normal_id', 'run_id', 'event_id'], sort=False).head(1)
    si_ecdf(a, terminal.operative_temperature_max_abs_difference_c, shared_BLUE)
    a.set_xlabel('Temperature residual in\nlast recovery hour (°C)')
    maximum = hourly.groupby(['normal_id', 'run_id', 'event_id']).operative_temperature_max_abs_difference_c.max()
    si_ecdf(b, maximum, shared_PURPLE)
    b.set_xlabel('Largest follow-on\ntemperature residual (°C)')
    si_ecdf(c, totals.facility_electricity_post_recovery_change_kwh, shared_GREEN)
    c.set_xlabel('Follow-on electricity\nchange (kWh)')
    for column, paint in [('facility_electricity_interval_mean_kw_difference', shared_BLUE), ('hvac_electricity_interval_mean_kw_difference', shared_PURPLE)]:
        quantiles = hourly.groupby('audit_hour_index')[column].quantile([0.05, 0.5, 0.95]).unstack()
        d.plot(np.arange(len(quantiles)), quantiles[0.5], color=paint, lw=1)
        d.fill_between(np.arange(len(quantiles)), quantiles[0.05], quantiles[0.95], color=paint, alpha=0.15, linewidth=0)
    d.set(xlabel='Hours after accounting end', ylabel='Action minus baseline (kW)')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Last recovery hour', 'Follow-on temperature', 'Six-hour electricity', 'Hourly electricity residual']):
        shared_title(ax, letter, name)
    lo, hi = d.get_ylim()
    d.set_ylim(lo, hi + 0.55 * (hi - lo))
    d.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['Whole building', 'HVAC'], loc='upper right', handlelength=0.8)
    return (fig, 'abcd')

si_FUNCTIONS = {10: si_s10, 11: si_s11, 12: si_s12}

extended_GROUPS = {4: (10, 11, 12)}

def extended_draw(number, output):
    global si_canvas
    fonts = shared_style()
    fig = plt.figure(figsize=(179 / 25.4, 210 / 25.4))
    source = si_Sources(ROOT / 'Data_for_Figures/output/from_raw', ROOT / 'Data_for_Figures/output/strengthening')
    panels, legends, colorbars, origins = ([], [], [], [])
    original_canvas = si_canvas

    def canvas(count=4, legend_gap=False):
        axes = [fig.add_axes([0.1, 0.1, 0.3, 0.2]) for _ in range(count)]
        panels.extend(axes)
        return (fig, axes)
    try:
        si_canvas = canvas
        for old_number in extended_GROUPS[number]:
            before_axes, before_legends, before_panels = (len(fig.axes), len(fig.legends), len(panels))
            si_FUNCTIONS[old_number](source)
            new_panels = panels[before_panels:]
            origins.extend(((old_number, letter) for letter in string.ascii_lowercase[:len(new_panels)]))
            legends.extend(((old_number, legend) for legend in fig.legends[before_legends:]))
            colorbars.extend(((old_number, ax) for ax in fig.axes[before_axes:] if ax not in new_panels))
    finally:
        si_canvas = original_canvas
    if number in (1, 3):
        order = [8, 9, 4, 5, 6, 7, 0, 1, 2, 3] if number == 1 else list(range(8))
        top = [[0.145, 0.765, 0.34, 0.18], [0.645, 0.765, 0.34, 0.18]]
        slots = top + [[x, y, 0.19, 0.145] for y in [0.525, 0.275] for x in [0.145, 0.465, 0.785]] + [[0.145, 0.055, 0.34, 0.145], [0.645, 0.055, 0.34, 0.145]]
        if number == 3:
            top = [[0.13, 0.765, 0.32, 0.18], [0.63, 0.765, 0.32, 0.18]]
            slots = top + [[x, 0.445, 0.19, 0.2] for x in [0.145, 0.465, 0.785]] + [[x, 0.115, 0.19, 0.23] for x in [0.145, 0.465, 0.785]]
    else:
        order = list(range(len(panels)))
        slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.565, 0.33, 0.095] for x in [0.145, 0.465, 0.785]]
        if number == 4:
            slots = [[x, y, 0.19, 0.13] for y in [0.8, 0.55, 0.32, 0.09] for x in [0.145, 0.465, 0.785]]
    panel_map = {}
    for letter, index, bounds in zip(string.ascii_lowercase, order, slots):
        ax = panels[index]
        ax.set_position(bounds)
        heading = ax.get_title(loc='left').split(' ', 1)[1]
        heading = heading.replace('\n', ': ')
        short = {'Later requests: Repeated calibration': 'Later requests, repeated', 'Acceptance: Repeated calibration': 'Acceptance, repeated', 'Later requests: Single-record calibration': 'Later requests, single', 'Acceptance: Single-record calibration': 'Acceptance, single', 'Individual minus building': 'Individual information', 'Center prediction error': 'Later-center error', 'Magnitude of exceedance': 'Service exceedance', 'Feasible settings out of 54': 'Service frontier', 'Annual operation support': 'Operation support', 'End-use relationship': 'Heating and cooling', 'Mean choices exceeding individual service limits': 'Service exceedances', 'Where averaging matters': 'Service exceedances', 'Hourly electricity residual': 'Electricity residual', 'Repeated-record support': 'Repeated records'}.get(heading, heading)
        renderer = fig.canvas.get_renderer()
        heading_width = renderer.get_text_width_height_descent(f'({letter}) {short}', FontProperties(family='Times New Roman', size=10, weight='bold'), False)[0]
        row_neighbors = [slot[0] for slot in slots if abs(slot[1] - bounds[1]) < 1e-10 and slot[0] > bounds[0]]
        heading_right = min(row_neighbors) - 0.012 if row_neighbors else 0.993
        available = (heading_right - bounds[0]) * fig.bbox.width
        if heading_width > available:
            words = short.split()
            middle = min(range(1, len(words)), key=lambda i: abs(len(' '.join(words[:i])) - len(' '.join(words[i:]))))
            short = ' '.join(words[:middle]) + '\n' + ' '.join(words[middle:])
        shared_title(ax, letter, short)
        panel_map[letter] = {'source_figure': origins[index][0], 'source_panel': origins[index][1]}
        if ax.get_xlabel() == 'Choice-disagreement frequency':
            ax.set_xlabel('Choice-disagreement\nfrequency')
        if ax.get_xlabel() == 'Reassignment allowance (%)':
            ax.set_xlabel('Reassignment\nallowance (%)')
        ax.tick_params(axis='both', pad=2)
        if ax.get_xscale() == 'linear' and len(ax.get_xticks()) > 6:
            ax.xaxis.set_major_locator(MaxNLocator(3))
        if origins[index][0] == 3 and origins[index][1] in 'abcd':
            ax.set_yticks(ax.get_yticks(), [t.get_text().replace(' + ', '\n+ ') for t in ax.get_yticklabels()])
        for coordinate, limits in [(ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())]:
            ticks = coordinate.get_majorticklocs()
            inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
            if not inside.all():
                coordinate.set_ticks(ticks[inside])
    for old_number, axis in colorbars:
        parent = panels[origins.index((old_number, 'c' if old_number == 5 else 'a'))]
        bounds = parent.get_position()
        if old_number == 5:
            parent.set_position([bounds.x0, bounds.y0, bounds.width - 0.035, bounds.height])
            bounds = parent.get_position()
        axis.set_position([bounds.x1 + 0.012, bounds.y0, 0.012, bounds.height])
    for old_number, legend in legends:
        if number == 2 and old_number == 6:
            panels[6].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper left', handlelength=0.8)
            legend.remove()
            continue
        legend.set_loc('center')
        legend.set_ncols(2 if len(legend.get_texts()) == 2 else 3)
        legend.set_bbox_to_anchor({1: (0.55, 0.018), 2: (0.55, 0.502 if old_number == 6 else 0.027), 3: (0.55, 0.695 if old_number == 8 else 0.035), 4: (0.55, 0.73)}[number], transform=fig.transFigure)
    if number == 1:
        panels[0].set_xlim(10, 55)
        legend = panels[0].get_legend()
        panels[0].legend(legend.legend_handles, [t.get_text() for t in legend.get_texts()], loc='upper right', handlelength=0.75, handletextpad=0.35, borderaxespad=0.25)
    if number == 4:
        panels[5].set_position([0.785, 0.535, 0.19, 0.145])
        panel = panels[6]
        panel.set_ylim(0, 1.6)
        panel.get_legend().set_loc('upper left')
        panel.tick_params(axis='x', pad=5)
        for line in panel.lines:
            if len(line.get_xdata()) == 2 and all((x == 0 for x in line.get_xdata())):
                line.set_ydata([0, 1 / 1.6])
        colorbars[0][1].set_yticks([0, 200])
    if number == 3:
        panels[7].set_xlim(0.94, 1.06)
        panels[7].set_ylim(0, 1.5)
        panels[7].get_legend().set_loc('upper right')
    if number == 2:
        panels[0].set_ylim(0, 100)
        lo, hi = panels[6].get_ylim()
        panels[6].set_ylim(lo, hi + 0.5 * (hi - lo))
    if number in (2, 4):
        matrix_axis = panels[2 if number == 2 else 4]
        values = matrix_axis.images[0].get_array()
        for row in range(values.shape[0]):
            for column in range(values.shape[1]):
                value = int(values[row, column])
                if value:
                    matrix_axis.text(column, row, str(value), ha='center', va='center', color='white' if value > values.max() * 0.6 else 'black')
    output = Path(output)
    if not output.resolve().is_relative_to(ROOT):
        raise ValueError('Publication output must remain inside the project')
    try:
        shared_export(fig, output, 200 + number, source.paths, string.ascii_lowercase[:len(panels)], fonts)
    finally:
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = output / f'Figure_{200 + number}.{extension}'
            if old.exists():
                old.replace(output / f'Figure_E{number:02}.{extension}')
    receipt_path = output / f'Figure_E{number:02}.json'
    receipt = json.loads(receipt_path.read_text())
    receipt.update({'panel_sources': panel_map, 'extended_plot_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), 'figure_width_in': 179 / 25.4, 'figure_height_in': 210 / 25.4})
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
    return output / f'Figure_E{number:02}.png'

# Read the data, draw this figure and save it.
path = extended_draw(4, OUTPUT / "extended")
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 1


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_s2(s):
    cities = s.read('results/observations/CITY_OBSERVATION_SUMMARY.csv', True)
    policies = s.read('results/observation_figures/POLICY_RANGE_POSITIONS.csv', True)
    seasons = s.read('results/seasonal_command/SEASONAL_PLACE_EFFECTS.csv', True)
    rooms = s.read('results/shared_room/BURDEN_THRESHOLD_RESULTS.csv', True)
    fig, (a, b, c, d) = si_canvas()
    for ax in (b, d):
        bounds = ax.get_position()
        ax.set_position([0.64, bounds.y0, 0.33, bounds.height])
    for archive, g in cities.groupby('archive'):
        a.scatter(g.observation_records, g.demand_center_p90_c - g.demand_center_p10_c, s=10, alpha=0.65, color=shared_BLUE if 'ASH' in archive.upper() else shared_PURPLE, label='ASHRAE' if 'ASH' in archive.upper() else 'CTCD')
    a.set(xscale='log', xlabel='City records', ylabel='Demand 10th–90th\npercentile span (°C)')
    shared_title(a, 'a', 'Variation within cities')
    a.set_ylim(0, 14)
    a.legend(loc='upper right', handlelength=0.7)
    p = policies[policies.griffiths.eq('g0.50')].pivot(index='policy', columns='position', values='proportion')
    short = {name: name.split()[0].replace('USA', 'United States') for name in p.index}
    left = np.zeros(len(p))
    for key, color in [('below', shared_BLUE), ('within', shared_GREEN), ('above', shared_CORAL)]:
        b.barh(np.arange(len(p)), 100 * p[key], left=left, color=color, height=0.65)
        left += 100 * p[key]
    b.set_yticks(range(len(p)), [short[n] for n in p.index])
    b.invert_yaxis()
    b.set(xlabel='Demand centers (%)', xlim=(0, 100))
    b.set_xticks([0, 50, 100])
    shared_title(b, 'b', 'Published temperature ranges')
    for source, g in seasons.groupby('demand_source'):
        c.scatter(g.cool_mean_c, g.warm_mean_c, s=10, alpha=0.7, color=shared_BLUE if 'ASH' in source.upper() else shared_PURPLE)
    lim = [min(seasons.cool_mean_c.min(), seasons.warm_mean_c.min()) - 1, max(seasons.cool_mean_c.max(), seasons.warm_mean_c.max()) + 1]
    c.plot(lim, lim, color='#777777', lw=0.6, ls='--')
    c.set(xlabel='Cool-season center (°C)', ylabel='Warm-season center (°C)', xlim=lim, ylim=lim)
    shared_title(c, 'c', 'Seasonal place contrasts')
    for dataset, g in rooms[rooms.minutes.eq(30)].groupby('dataset'):
        d.plot(g.threshold, 100 * g.false_acceptance_windows / g.windows, marker='o', ms=3, lw=1, label='Singapore' if dataset == 'NUS' else 'Independent')
    d.set(xlabel='Mean-request threshold', ylabel='Windows with hidden\nindividual requests (%)')
    shared_title(d, 'd', 'Averaging shared requests')
    d.legend(loc='upper right', handlelength=0.8)
    d.set_ylim(0, 19)
    b.set_ylim(len(p) - 0.5, -2.3)
    b.legend([Patch(fc=x) for x in [shared_BLUE, shared_GREEN, shared_CORAL]], ['Below', 'Within', 'Above'], loc='upper center', ncol=3, columnspacing=0.6, handlelength=0.8)
    c.legend([Line2D([], [], color=shared_BLUE), Line2D([], [], color=shared_PURPLE)], ['ASHRAE', 'CTCD'], loc='lower right')
    return (fig, 'abcd')

si_FUNCTIONS = {2: si_s2}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "2", "--published-number", "1",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S01.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 2


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

si_CONTROL = {'uniform_average': 'Average', 'personal_center': 'Personal center', 'personal_range': 'Personal range'}

si_COLORS = {'uniform_average': shared_GOLD, 'personal_center': shared_BLUE, 'personal_range': shared_PURPLE}

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_s4(s):
    profiles = s.read('results/observation_figures/DEMAND_COMPOSITION_PROFILES.csv', True)
    profiles = profiles[profiles.context.eq('Miami')]
    domain = s.read('results/target_domain_final/TARGET_DOMAIN_SUMMARY.csv', True)
    anchors = s.read('results/global_final/GLOBAL_25C_SERVICE_ANCHORS.csv', True)
    fig, (a, b, c, d) = si_canvas()
    fig.set_size_inches(shared_WIDTH, 5.1)
    for ax, x, y in [(a, 0.13, 0.6), (b, 0.62, 0.6), (c, 0.13, 0.15), (d, 0.62, 0.15)]:
        ax.set_position([x, y, 0.35, 0.31])
    for ax, column, paint, label in [(a, 'centre_c', shared_BLUE, 'Demand center (°C)'), (b, 'range_width_c', shared_PURPLE, 'Personal range width (°C)')]:
        values = [g[column].to_numpy() for _, g in profiles.groupby('composition_id')]
        violin = ax.violinplot(values, positions=np.arange(1, 7), showextrema=False)
        for body in violin['bodies']:
            body.set_facecolor(paint)
            body.set_edgecolor('none')
            body.set_alpha(0.65)
        ax.scatter(range(1, 7), [np.median(v) for v in values], s=12, color='black', zorder=3)
        ax.set(xlabel='Demand composition', ylabel=label)
        ax.set_xticks(range(1, 7))
    shared_title(a, 'a', 'Six demand compositions')
    shared_title(b, 'b', 'Range-width distributions')
    for control, g in domain.groupby('control'):
        c.scatter(g.base_target_min_c, g.base_target_max_c, s=14, alpha=0.6, color=si_COLORS[control], label=si_CONTROL[control])
    c.set(xlabel='Lowest zone target (°C)', ylabel='Highest zone target (°C)', xlim=(19.5, 27.5), ylim=(19.5, 34.5))
    c.plot([20, 27], [20, 27], color='#999999', lw=0.6, ls=':')
    shared_title(c, 'c', 'Target extent by control')
    for climate, g in anchors.groupby('climate'):
        d.scatter(g.anchor_mean_mismatch_c, g.anchor_tail_mismatch_c, s=14, color=shared_GREEN, alpha=0.7)
    d.set(xlabel='Population reference\nmismatch (°C)', ylabel='Least-served reference\nmismatch (°C)')
    shared_title(d, 'd', 'Reference service limits')
    c.legend(loc='upper left', handlelength=0.8)
    return (fig, 'abcd')

si_FUNCTIONS = {4: si_s4}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "4", "--published-number", "2",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S02.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 3


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s13(s):
    direct = s.read('results/global_final/GLOBAL_DIRECT_COMPARISONS.csv', True)
    matched = s.read('results/matched_service_final/MATCHED_REALIZED_SERVICE_TWO_METRIC_TOLERANCE_SUMMARY.csv', True)
    complete = s.read('results/energy_boundary_final/GLOBAL_ENERGY_BOUNDARY_DIRECT_COMPARISONS.csv', True)
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
        g = direct[direct.alternative_control.eq(control)]
        a.scatter(-g.delta_mean_reference_mismatch_c, g.delta_facility_purchased_pct, s=13, color=paint, alpha=0.6)
        b.scatter(-g.delta_fixed_tail_reference_mismatch_c, g.delta_facility_purchased_pct, s=13, color=paint, alpha=0.6)
        t = matched[matched.alternative_control.eq(control)].sort_values('tolerance_c')
        c.plot(t.tolerance_c, t.matched_context_control_groups, marker='o', ms=3, color=paint, lw=1)
        z = complete[complete.alternative_control.eq(control)]
        d.scatter(z.delta_complete_facility_kwh / 1000, z.delta_complete_facility_carbon_kg / 1000, s=13, color=paint, alpha=0.6)
    for ax in [a, b, d]:
        ax.axhline(0, color='#aaaaaa', ls=':', lw=0.6)
        ax.axvline(0, color='#aaaaaa', ls=':', lw=0.6)
    a.set(xlabel='Population mismatch\nreduction (°C)', ylabel='Purchased-energy change (%)')
    b.set(xlabel='Least-served mismatch\nreduction (°C)', ylabel='Purchased-energy change (%)')
    c.set(xlabel='Service-matching tolerance (°C)', ylabel='Matched comparisons', ylim=(0, 56))
    d.set(xlabel='Annual energy change (MWh)', ylabel='Annual operational\ncarbon change (t CO₂)')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Direct population service', 'Direct tail service', 'Matching both services', 'Annual carbon inventory']):
        shared_title(ax, letter, name)
    si_common_legend(fig, ['Personal center', 'Personal range'], [shared_BLUE, shared_PURPLE], y=0.5)
    return (fig, 'abcd')

si_FUNCTIONS = {13: si_s13}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "13", "--published-number", "3",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S03.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 4


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_ecdf(ax, values, color, label=None):
    values = np.sort(pd.Series(values).dropna().to_numpy())
    ax.step(values, np.arange(1, len(values) + 1) / len(values), where='post', color=color, lw=1.2, label=label)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cumulative fraction')

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s14(s):
    annual = s.read('results/seasonal_facility_energy_final/SEASONAL_COMMAND_ANNUAL_ENERGY_COMPLETE_FACILITY.csv', True)
    seasons = s.read('results/seasonal_facility_energy_final/SEASONAL_COMMAND_SEASON_METRICS_COMPLETE_FACILITY.csv', True)
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    for fraction, ax in [(0.5, a), (1, b)]:
        g = annual[annual.seasonal_contrast_fraction.eq(fraction)]
        baseline = g[g.control.eq('uniform_average')].set_index(['climate', 'composition_id']).facility_purchased_kwh
        for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
            h = g[g.control.eq(control)].set_index(['climate', 'composition_id'])
            change = (h.facility_purchased_kwh / baseline.reindex(h.index) - 1) * 100
            si_ecdf(ax, change, paint)
        ax.set_xlabel('Annual energy change (%)')
    g = seasons[seasons.seasonal_contrast_fraction.eq(1)]
    keys = ['climate', 'composition_id', 'local_season']
    baseline = g[g.control.eq('uniform_average')].set_index(keys)
    for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
        h = g[g.control.eq(control)].set_index(keys)
        ref = baseline.reindex(h.index)
        c.scatter(ref.mean_reference_mismatch_c - h.mean_reference_mismatch_c, (h.facility_purchased_kwh - ref.facility_purchased_kwh) / 1000, s=8, color=paint, alpha=0.4)
        d.scatter(ref.fixed_tail_reference_mismatch_c - h.fixed_tail_reference_mismatch_c, (h.facility_purchased_kwh - ref.facility_purchased_kwh) / 1000, s=8, color=paint, alpha=0.4)
    c.set(xlabel='Seasonal population\nmismatch reduction (°C)', ylabel='Seasonal energy change (MWh)')
    d.set(xlabel='Seasonal least-served\nmismatch reduction (°C)', ylabel='Seasonal energy change (MWh)')
    for ax in [c, d]:
        ax.axhline(0, color='#aaaaaa', ls=':', lw=0.6)
        ax.axvline(0, color='#aaaaaa', ls=':', lw=0.6)
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Half seasonal contrast', 'Full seasonal contrast', 'Seasonal population service', 'Seasonal tail service']):
        shared_title(ax, letter, name)
    si_common_legend(fig, ['Personal center', 'Personal range'], [shared_BLUE, shared_PURPLE], y=0.5)
    return (fig, 'abcd')

si_FUNCTIONS = {14: si_s14}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "14", "--published-number", "4",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S04.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 5


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s15(s):
    permission = s.read('results/spatial_final/GLOBAL_PERMISSION_COMPARISONS.csv', True)
    placement = s.read('results/spatial_final/GLOBAL_PLACEMENT_COMPARISONS.csv', True)
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    for frame, axes in [(permission, (a, b)), (placement, (c, d))]:
        for control, paint in [('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
            g = frame[frame.control.eq(control)]
            axes[0].scatter(-g.delta_mean_reference_mismatch_c, -g.delta_fixed_tail_reference_mismatch_c, color=paint, s=8, alpha=0.45)
            axes[1].scatter(-g.delta_mean_reference_mismatch_c, g.delta_facility_purchased_pct, color=paint, s=8, alpha=0.45)
        axes[0].set(xlabel='Population mismatch\nreduction (°C)', ylabel='Least-served mismatch\nreduction (°C)')
        axes[1].set(xlabel='Population mismatch\nreduction (°C)', ylabel='Purchased-energy change (%)')
    for ax, letter, name in zip([a, b, c, d], 'abcd', ['Command-scope service', 'Command-scope energy', 'Ordered-placement service', 'Ordered-placement energy']):
        shared_title(ax, letter, name)
    si_common_legend(fig, ['Personal center', 'Personal range'], [shared_BLUE, shared_PURPLE], y=0.5)
    return (fig, 'abcd')

si_FUNCTIONS = {15: si_s15}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "15", "--published-number", "5",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S05.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))


## Fig. S 6


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import hashlib
import re
import matplotlib
matplotlib.use("Agg")
import hashlib
import json
from pathlib import Path
import matplotlib as mpl
import numpy as np
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.legend import Legend
import argparse
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.patches import Patch
from matplotlib.text import Text
import pandas as pd
from matplotlib.font_manager import FontProperties
from matplotlib import font_manager
from matplotlib.patches import Patch, Polygon, Rectangle

# Figure settings: edit these values and the panel code below.
FONT_SIZE = 9
PANEL_TITLE_SIZE = 10
DPI = 300
ROOT = Path.cwd().resolve()
if ROOT.name == "plot":
    ROOT = ROOT.parent
__file__ = str(ROOT / "plot/Plot.ipynb")
OUTPUT = Path(globals().get("PUBLICATION_OUTPUT", ROOT / "output/publication"))
DATA_ROOT = next(p for p in [ROOT / "Data", ROOT / "Source_Data", ROOT / "Data_for_Figures"] if (p / "index.json").is_file() or (p / "FIGURE_DATA_MANIFEST.json").is_file())
_index_path = DATA_ROOT / ("index.json" if (DATA_ROOT / "index.json").is_file() else "FIGURE_DATA_MANIFEST.json")
_index = json.loads(_index_path.read_text(encoding="utf-8"))
DATA_FILES = _index if isinstance(_index, list) else _index["files"]


def source_key(value):
    value = str(value).replace("\\", "/")
    value = value.split("Data_for_Figures/")[-1]
    value = value.replace("output/from_raw/", "baseline/").replace("output/strengthening/", "analysis/")
    return re.sub(r"run_[^/]+/", "", value)

def data_path(request):
    key = source_key(request)
    exact = [r for r in DATA_FILES if source_key(r.get("key", r.get("result", ""))) == key]
    matches = exact or [r for r in DATA_FILES if Path(r.get("key", r.get("result", ""))).name == Path(key).name]
    if len(matches) == 1:
        return ROOT / matches[0].get("file", matches[0].get("result"))
    for path in [DATA_ROOT / "Maps" / Path(key).name, ROOT / "src/thermal_heterogeneity/assets" / Path(key).name]:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Expected one source dataset for {request}; found {len(matches)}")


# Data, styles, panel layout and drawing routines for this figure.

base_FONT_FAMILY = 'Times New Roman'

class base_FigureContractError(RuntimeError):
    """Raised when a rendered figure violates the locked visual contract."""

class base_FigureFontError(base_FigureContractError):
    """Raised when exact Times New Roman would be replaced by another font."""

def base_require_times_new_roman() -> dict[str, str]:
    """Resolve the exact normal and bold fonts, failing instead of falling back."""
    local_fonts = Path(__file__).resolve().parents[2] / 'tools/fonts'
    if local_fonts.is_dir():
        from matplotlib import font_manager
        font_manager.fontManager.ttflist = [font for font in font_manager.fontManager.ttflist if font.name != base_FONT_FAMILY]
        for font in sorted(local_fonts.glob('times*.ttf')):
            font_manager.fontManager.addfont(str(font))
        font_manager.fontManager._findfont_cached.cache_clear()
    resolved: dict[str, str] = {}
    for weight in ('normal', 'bold'):
        properties = FontProperties(family=[base_FONT_FAMILY], weight=weight)
        try:
            font_path = Path(findfont(properties, fallback_to_default=False))
        except ValueError as exc:
            raise base_FigureFontError(f'{base_FONT_FAMILY} ({weight}) is required; install it before rendering') from exc
        actual_name = FontProperties(fname=str(font_path)).get_name()
        if actual_name != base_FONT_FAMILY:
            raise base_FigureFontError(f'font fallback prohibited: requested {base_FONT_FAMILY}, resolved {actual_name}')
        resolved[weight] = str(font_path.resolve())
    return resolved

def base_audit_data_safe_legends(figure: Figure) -> dict[str, int]:
    """Verify all legends are transparent and reserved-band legends avoid data."""
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    legends_checked = 0
    data_safe_checked = 0
    for axis in figure.axes:
        legend = axis.get_legend()
        if legend is None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
        if legend.get_gid() != 'thermal-heterogeneity-data-safe-legend':
            continue
        data_safe_checked += 1
        original_ymax = getattr(legend, '_thermal_original_ymax', None)
        if original_ymax is None:
            raise base_FigureContractError('data-safe legend lacks its reserved-band boundary')
        data_ceiling = axis.transData.transform((0, original_ymax))[1]
        if legend.get_window_extent(renderer).y0 < data_ceiling - 0.5:
            raise base_FigureContractError('data-safe legend extends below its reserved top band')
    for legend in figure.findobj(Legend):
        if legend.axes is not None:
            continue
        legends_checked += 1
        if legend.get_frame().get_visible():
            raise base_FigureContractError('legend frame/background must be disabled')
    return {'legends_checked': legends_checked, 'data_safe_legends_checked': data_safe_checked}

shared_FONT = FONT_SIZE

shared_TITLE = PANEL_TITLE_SIZE

shared_WIDTH = 8306 / 1440

shared_BLUE, shared_PURPLE, shared_GREEN, shared_CORAL, shared_GOLD = ('#6B9CBF', '#AA8BAE', '#70A598', '#CC8176', '#C9AE70')

def shared_style():
    mpl.rcdefaults()
    fonts = base_require_times_new_roman()
    mpl.rcParams.update({'font.family': 'Times New Roman', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Times New Roman', 'mathtext.it': 'Times New Roman:italic', 'mathtext.bf': 'Times New Roman:bold', 'font.size': shared_FONT, 'axes.titlesize': shared_TITLE, 'axes.titleweight': 'bold', 'axes.labelsize': shared_FONT, 'axes.labelpad': 4, 'xtick.labelsize': shared_FONT, 'ytick.labelsize': shared_FONT, 'xtick.major.pad': 3, 'ytick.major.pad': 3, 'text.color': 'black', 'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.edgecolor': 'black', 'axes.linewidth': 0.6, 'legend.fontsize': shared_FONT, 'legend.title_fontsize': shared_FONT, 'legend.frameon': False, 'legend.labelspacing': 0.45, 'legend.facecolor': 'none', 'legend.edgecolor': 'none', 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'svg.hashsalt': 'HEM-strengthened-publication', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
    return fonts

def shared_title(ax, letter, text):
    label = ax.set_title(f'({letter}) {text}', loc='left', pad=6, fontsize=shared_TITLE, fontweight='bold', color='black')
    label.set_gid('thermal-heterogeneity-panel-title')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=3, width=0.6)

def shared_export(fig, output, number, paths, letters, fonts):
    fig.canvas.draw()
    for text in fig.findobj(Text):
        if not text.get_visible() or not text.get_text().strip():
            continue
        expected = shared_TITLE if text.get_gid() == 'thermal-heterogeneity-panel-title' else shared_FONT
        actual = FontProperties(fname=findfont(text.get_fontproperties(), fallback_to_default=False)).get_name()
        if actual != 'Times New Roman' or abs(text.get_fontsize() - expected) > 1e-08:
            raise ValueError(f'Font contract differs: {text.get_text()!r}')
    renderer = fig.canvas.get_renderer()
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip()]
    boxes = [Text.get_window_extent(t, renderer) for t in texts]
    collision = []
    for i, first in enumerate(boxes):
        for j in range(i + 1, len(boxes)):
            second = boxes[j]
            width = min(first.x1, second.x1) - max(first.x0, second.x0)
            height = min(first.y1, second.y1) - max(first.y0, second.y0)
            if width > 0.5 and height > 0.5:
                collision.append([texts[i].get_text(), texts[j].get_text()])
    clipped = [t.get_text() for t, box in zip(texts, boxes) if box.x0 < 0 or box.y0 < 0 or box.x1 > fig.bbox.width or (box.y1 > fig.bbox.height)]
    marker_overlap = []
    if number == 1:
        map_axis = fig.axes[0]
        for collection in map_axis.collections:
            if not isinstance(collection, PathCollection) or not len(collection.get_offsets()):
                continue
            positions = collection.get_offset_transform().transform(collection.get_offsets())
            sizes = np.broadcast_to(collection.get_sizes(), (len(positions),))
            for text, box in zip(texts, boxes):
                if text.axes is not map_axis:
                    continue
                radius = np.sqrt(sizes) * fig.dpi / 144
                hit = (positions[:, 0] + radius > box.x0) & (positions[:, 0] - radius < box.x1) & (positions[:, 1] + radius > box.y0) & (positions[:, 1] - radius < box.y1)
                if np.any(hit):
                    marker_overlap.append(text.get_text())
    legends = base_audit_data_safe_legends(fig)
    output.mkdir(parents=True, exist_ok=True)
    prefix = output / f'Figure_{number:02}'
    for extension in ('pdf', 'svg', 'png'):
        fig.savefig(prefix.with_suffix('.' + extension), dpi=300, facecolor='white')
    audit = {'status': 'requires_layout_correction' if collision or clipped or marker_overlap else 'rendered_pending_visual_review', 'font_pt': shared_FONT, 'title_pt': shared_TITLE, 'figure_width_in': shared_WIDTH, 'fonts': fonts, 'panels': letters, 'text_collisions': collision, 'clipped_text': clipped, 'map_text_marker_overlaps': marker_overlap, 'legends': legends, 'input_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    prefix.with_suffix('.json').write_text(json.dumps(audit, indent=2) + '\n')
    plt.close(fig)
    print(json.dumps({'figure': number, 'status': audit['status'], 'collision': collision, 'clipped': clipped, 'marker_overlap': marker_overlap}))
    if collision or clipped or marker_overlap:
        raise ValueError('Figure layout requires correction; preview and diagnostics retained')

si_CONTROL = {'uniform_average': 'Average', 'personal_center': 'Personal center', 'personal_range': 'Personal range'}

class si_Sources:

    def __init__(self, raw, analysis):
        self.raw, self.analysis, self.paths = (raw, analysis, [])

    def read(self, name, raw=False):
        path = data_path(('baseline/' if raw else 'analysis/') + name)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

    def latest(self, folder, filename):
        path = data_path('analysis/' + folder + '/' + filename)
        self.paths.append(path)
        return pd.read_csv(path, low_memory=False)

def si_canvas(count=4, legend_gap=False):
    rows = count // 2
    height = 5.5 if rows == 2 else 7.0
    fig = plt.figure(figsize=(shared_WIDTH, height))
    positions = ([0.625, 0.135] if legend_gap else [0.61, 0.15]) if rows == 2 else [0.745, 0.44, 0.135]
    axes = [fig.add_axes([x, y, 0.3, 0.29 if rows == 2 else 0.19]) for y in positions for x in [0.17, 0.67]]
    return (fig, axes)

def si_common_legend(fig, labels, colors, y=0.035, columns=None):
    fig.legend([Patch(facecolor=c) for c in colors], labels, loc='center', bbox_to_anchor=(0.56, y), ncol=columns or len(labels), columnspacing=1, handlelength=1, handletextpad=0.4)

def si_s16(s):
    x = s.read('results/response_mean_decomposition/RESPONSE_MEAN_DECOMPOSITION.csv', True)
    fig, (a, b, c, d) = si_canvas(legend_gap=True)
    total = x[x.group_dimension.eq('overall')].set_index('control').loc[list(si_CONTROL)]
    positions = np.arange(3)
    for ax, col, name in [(a, 'raw_positive_zero_filled_mean_kw', 'Physical response'), (b, 'service_preserving_zero_filled_mean_kw', 'Service-preserving power')]:
        ax.bar(positions, total[col], color=[shared_GOLD, shared_BLUE, shared_PURPLE], width=0.6)
        ax.set_xticks(positions, ['Average', 'Personal\ncenter', 'Personal\nrange'])
        ax.set_ylabel('Fraction of cases' if ax is c else 'Mean event power (kW)')
        shared_title(ax, 'abcd'[[a, b, c, d].index(ax)], name)
    climates = x[x.group_dimension.eq('climate')]
    for control, paint in [('uniform_average', shared_GOLD), ('personal_center', shared_BLUE), ('personal_range', shared_PURPLE)]:
        g = climates[climates.control.eq(control)]
        c.scatter(g.qualified_positive_fraction, g.qualified_positive_conditional_mean_kw, color=paint, s=18)
    c.set(xlabel='Positive-response fraction', ylabel='Conditional power (kW)', xlim=(-0.04, 1.04))
    shared_title(c, 'c', 'Frequency and intensity')
    actions = x[x.group_dimension.eq('action_id')].pivot(index='group_value', columns='control', values='service_preserving_zero_filled_mean_kw').reindex(columns=list(si_CONTROL))
    action_names = {'uniform_0.5': 'Uniform 0.5 °C', 'uniform_1.0': 'Uniform 1 °C', 'protected_1.0': 'Protected 1 °C', 'precondition_0.5_uniform_1.0': 'Precondition +\nuniform', 'precondition_0.5_protected_1.0': 'Precondition +\nprotected'}
    image = d.imshow(actions.to_numpy(), aspect='auto', origin='upper', cmap='Purples')
    d.set_xticks(range(3), ['Average', 'Personal\ncenter', 'Personal\nrange'])
    d.set_yticks(range(len(actions)), [action_names[n] for n in actions.index])
    shared_title(d, 'd', 'Action-specific power')
    d.set_position([0.67, 0.165, 0.3, 0.26])
    colorbar = fig.colorbar(image, cax=fig.add_axes([0.67, 0.065, 0.3, 0.014]), orientation='horizontal')
    colorbar.set_label('All-case mean power (kW)', labelpad=0)
    si_common_legend(fig, list(si_CONTROL.values()), [shared_GOLD, shared_BLUE, shared_PURPLE], y=0.5)
    return (fig, 'abcd')

si_FUNCTIONS = {16: si_s16}

def si_main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--raw-root', type=Path, default=ROOT / 'output/from_raw')
    parser.add_argument('--analysis-root', type=Path, default=ROOT / 'output/strengthening')
    parser.add_argument('--output-root', type=Path, default=ROOT / 'output/publication/supplementary')
    parser.add_argument('--figure', type=int, nargs='+', default=list(si_FUNCTIONS))
    parser.add_argument('--published-number', type=int)
    args = parser.parse_args(argv)
    fonts = shared_style()
    if not all((p.resolve().is_relative_to(ROOT) for p in [args.raw_root, args.analysis_root, args.output_root])):
        raise ValueError('All publication inputs and outputs must remain inside the project')
    for number in args.figure:
        published = args.published_number or number
        if args.published_number and len(args.figure) != 1:
            raise ValueError('A published number requires exactly one source figure')
        source = si_Sources(args.raw_root, args.analysis_root)
        fig, letters = si_FUNCTIONS[number](source)
        for axis in fig.axes:
            for coordinate, limits in [(axis.xaxis, axis.get_xlim()), (axis.yaxis, axis.get_ylim())]:
                ticks = coordinate.get_majorticklocs()
                inside = (ticks >= min(limits) - 1e-10) & (ticks <= max(limits) + 1e-10)
                if not inside.all():
                    coordinate.set_ticks(ticks[inside])
        shared_export(fig, args.output_root, 100 + published, source.paths, letters, fonts)
        for extension in ['png', 'pdf', 'svg', 'json']:
            old = args.output_root / f'Figure_{100 + published}.{extension}'
            old.replace(args.output_root / f'Figure_S{published:02}.{extension}')
        path = args.output_root / f'Figure_S{published:02}.json'
        receipt = json.loads(path.read_text())
        receipt['supplementary_plot_sha256'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
        path.write_text(json.dumps(receipt, indent=2) + '\n')

# Read the data, draw this figure and save it.
si_main(["--figure", "16", "--published-number", "6",
    "--raw-root", str(ROOT / "Data_for_Figures/output/from_raw"),
    "--analysis-root", str(ROOT / "Data_for_Figures/output/strengthening"),
    "--output-root", str(OUTPUT / "supplementary")])
path = OUTPUT / "supplementary/Figure_S06.png"
try:
    from IPython.display import display, Image as DisplayImage
    get_ipython
except (ImportError, NameError):
    print(path)
else:
    display(DisplayImage(filename=str(path)))
